# ANÁLISIS DE LOS COSTOS DE PROCESO DE PRODUCCIÓN

## Extracción y limpieza de los datos

In [1]:
# Importamos las librerías necesarias para el desarrollo del código
import pandas as pd
import plotly.express as px
import re
import unicodedata
import plotly.graph_objects as go
from dash import Dash, html, dcc, Input, Output
app = Dash(__name__)

# Primer Excel utilizado para todo el análisis inicial
df_costos = pd.read_excel('df_final.xlsx')

# Segundo excel con categorías organizadas y fechas desde 2023
df_costos_fechacom = pd.read_excel('df_final1.xlsx')

# Tercer excel con fechas desde 2023 y valor completo de la op pero sin categorías
df_costos_valorop = pd.read_excel('df_fin.xlsx')


In [3]:
# Definimos la lista de categorías oficiales para la clasificación de productos
categorias_oficiales = [
    'Accesorios', 'Bata', 'Bermuda', 'Blusa', 'Bolígrafo', 'Botas', 'Botella', 'Botón', 'Broche', 'Buzo', 'Camibuzo', 
    'Camiseta', 'Camisa', 'Cachuchera', 'Casco', 'Chaleco', 'Chaqueta', 'Cinta', 'Cofia', 'Conjunto', 'Cordón', 
    'Cremallera', 'Cuello', 'Delantal', 'Diseño', 'Embone', 'Falda', 'Gafas seguridad', 'Gorra', 'Gorro', 'Guantes', 
    'Guata', 'Hilo', 'Hoodie', 'Impermeable', 'Jean', 'Libreta', 'Lonchera', 'Mangas', 'Marquilla', 'Máscara', 'Medias', 
    'Morral', 'Ojalete', 'Overol', 'Pantalón', 'Pantaloneta', 'Pañoleta', 'Paraguas', 'Pavas', 'Polainas', 'Resorte', 
    'Ropa interior', 'Saco', 'Sastre', 'Servicios', 'Sesgo', 'Sudadera', 'Suéter', 'Tankas', 'Tapabocas', 
    'Tapaoídos', 'Tecnología', 'Termo', 'Uniformes', 'Velcro', 'Zapatos', 'Escafandra'
]

# Creamos una función para normalizar el texto, eliminando acentos y convirtiendo a minúsculas
def normalizar(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    return texto

# Creamos una función que clasifique los productos en las categorías oficiales.
def clasificar(Producto):
    producto_norm = normalizar(Producto)
    # Recorremos cada categoría oficial y verificamos si alguna de las palabras está presente en el nombre del producto
    for cat in categorias_oficiales:
        cat_norm = normalizar(cat)
        pattern = r'\b' + re.escape(cat_norm) + r'\b'
        # asignamos la categoría correspondiente si encontramos una coincidencia
        if re.search(pattern, producto_norm):
            return cat
    # Si no encontramos coincidencia, asignamos 'Otros'
    return 'Otros'

df_costos['Categoría'] = df_costos['Producto'].apply(clasificar)   
#print(df_costos['Categoría'].value_counts())
df_costos['Proceso Producción'].unique()

paleta_molt_oscura = ["#29104A", "#4A1B8C", "#0F0530", "#3C1053", "#5C2B92"]
paleta_contraste_oscuro = ["#3C1053", "#003F5C", "#005F4B", "#6B2D3A", "#2F4F4F"]
paleta_textil = ["#5B46DF", "#4E6E58", "#A0522D", "#607B8B", "#8B4513"]

In [2]:
df_costos_fechacom.columns

Index(['Num-OP', 'OP Det', 'Producto', 'Cliente', 'Cantidad Solicitada', 'Id.',
       'Costeo_Producto', 'Fecha', 'costo_antes_iva', 'Proceso Producción',
       'OP-D', 'secuencia', 'OS-Proveedor', 'OS-Valor Unitario', 'OS-Cantidad',
       'Costeo', 'Nombre del Comercial', 'Diferencia valor unitario',
       'Valor total OS', 'valor total op', 'Diferencia', 'Categoría'],
      dtype='object')

## KPI

In [5]:
ahorro_total = abs(df_costos.loc[df_costos['Diferencia'] < 0, 'Diferencia'].sum())
perdida_total = df_costos.loc[df_costos['Diferencia'] > 0, 'Diferencia'].sum()
balance_total = df_costos['Diferencia'].sum()

ahorro_total_fmt = f"{ahorro_total:,.0f}"
perdida_total_fmt = f"{perdida_total:,.0f}"
balance_total_fmt = f"{balance_total:,.0f}"


## Atípicos

In [6]:
# Sacamos un top10 de las categorías que más cuestan en general
top_10_cats = (
    df_costos_fechacom.groupby('Categoría')['Valor total OS']
    .sum()
    .nlargest(10)
    .index
)
df_atipicos = df_costos_fechacom[df_costos_fechacom['Categoría'].isin(top_10_cats)]

fig_box = px.box(
    df_atipicos, 
    x='Categoría', 
    y='Diferencia valor unitario', 
    color='Categoría',   
    points='outliers',          
    hover_data=['Num-OP', 'OS-Proveedor', 'Cliente'], # Info extra al pasar el mouse
    labels={'OS-Valor Unitario': 'Costo Unitario ($)'}
)

fig_box.show()

## Pareto

### Pareto de proveedor

In [7]:
df_costos_fechacom['Fecha'] = pd.to_datetime(df_costos_fechacom['Fecha'])

# Filtro de los últimos 6 meses
df_filtrado = df_costos_fechacom[df_costos_fechacom['Fecha'] >= '2025-10-17'].copy()

# Creamos una variable para agrupar las tres columnas y poder comparar sobre los mismos datos
group_cols = ['Categoría', 'Cliente', 'Proceso Producción']

# Creamos un df agrupando las columnas del paso anterior y el valor de la OS, sacando entre estos datos solo el valor más bajo de la misma agrupacion
df_minimos = df_filtrado.groupby(group_cols)['OS-Valor Unitario'].transform('min')

# Del df original (filtrado) creamos una columna de diferencia para ver los valores que son diferentes al minimo que sacamos anteriormente
df_filtrado['Diferencia_Unitaria'] = df_filtrado['OS-Valor Unitario'] - df_minimos

# Multiplicamos la columna de diferencia por la cantidad para saber cuánto a nivel general se pagó de más 
df_filtrado['Ahorro_Potencial_Total'] = df_filtrado['Diferencia_Unitaria'] * df_filtrado['OS-Cantidad']

# Agrupación para el gráfico
df_pareto_ahorro = df_filtrado.groupby('OS-Proveedor').agg({
    'Ahorro_Potencial_Total': 'sum',
    'Valor total OS': 'sum',
    'OS-Cantidad': 'sum'
}).reset_index()


df_pareto_ahorro = df_pareto_ahorro.sort_values(by='Ahorro_Potencial_Total', ascending=False)

# Cálculo de acumulados
total_ahorro_posible = df_pareto_ahorro['Ahorro_Potencial_Total'].sum()
df_pareto_ahorro['Porcentaje'] = (df_pareto_ahorro['Ahorro_Potencial_Total'] / total_ahorro_posible) * 100
df_pareto_ahorro['Acumulado'] = df_pareto_ahorro['Porcentaje'].cumsum()


fig_ahorro = go.Figure()

fig_ahorro.add_trace(go.Bar(
    x=df_pareto_ahorro['OS-Proveedor'],
    y=df_pareto_ahorro['Ahorro_Potencial_Total'],
    name='Gasto Excedente ($)',
    marker_color='#EF553B',
    customdata=df_pareto_ahorro[['Valor total OS', 'OS-Cantidad']], 
    hovertemplate=(
        "<b>Proveedor: %{x}</b><br>" +
        "Ahorro potencial total: %{y:$,.0f}<br>" +
        "Valor total de OS: %{customdata[0]:$,.0f}<br>" + 
        "Total Prendas: %{customdata[1]:,.0f}" +
        "<extra></extra>"
    )
))


fig_ahorro.add_trace(go.Scatter(
    x=df_pareto_ahorro['OS-Proveedor'],
    y=df_pareto_ahorro['Acumulado'],
    name='% Acumulado',
    yaxis='y2',
    line=dict(color='#636EFA', width=3),
    hovertemplate="Progreso Acumulado: %{y:.2f}%<extra></extra>"
))

# 7. Configuración de Layout
fig_ahorro.update_layout(
    xaxis=dict(title='Proveedor'),
    yaxis=dict(title='Ahorro Potencial Acumulado ($)'),
    yaxis2=dict(title='Porcentaje (%)', overlaying='y', side='right', range=[0, 105]),
    hovermode='x unified'
)

# Línea del 80% (Los proveedores que causan el 80% del sobrecosto)
fig_ahorro.add_shape(type="line", x0=0, x1=1, y0=80, y1=80, yref='y2', xref='paper',
                     line=dict(color="black", width=2, dash="dot"))

fig_ahorro.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=10,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_ahorro.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_ahorro.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_ahorro.show()

In [8]:
df_filtrado[group_cols + ['OS-Valor Unitario', 'OP Det']]

,Categoría,Cliente,Proceso Producción,OS-Valor Unitario,OP Det
0,Pantalón,SITARA,Confección,5000.0,6668-1-PANTALON RESORTADO CABALLERO
1,Camisa,MINISO,Estampado,2400.0,6656-1-POLO MANGA LARGA
2,Camisa,MINISO,Estampado,2400.0,6656-2-POLO MANGA LARGA
3,Saco,Clínica Palermo,Paquete Completo Sin Marca,62000.0,6658-13-Saco Tejido
4,Saco,Clínica Palermo,Paquete Completo Sin Marca,6200.0,6658-16-Saco Tejido
...,...,...,...,...,...
5140,Buzo,SITARA,Confección,8500.0,6597-2-HODDIE CERRADO
5141,Buzo,SITARA,Confección,5000.0,6573-1-PANTALON RESORTADO CABALLERO
5142,Buzo,SITARA,Bordado,3500.0,6573-3-HODDIE CERRADO
5143,Buzo,SITARA,Confección,6500.0,6573-3-HODDIE CERRADO


In [9]:
df_minimos

0        5000.0
1        2400.0
2        2400.0
3        6200.0
4        6200.0
          ...  
5140     5000.0
5141     5000.0
5142      800.0
5143     5000.0
11082     800.0
Name: OS-Valor Unitario, Length: 682, dtype: float64

In [10]:
# Filtro de fecha para tomar exactamente un año 
df_filtrado_pro = df_costos_valorop[df_costos_valorop['Fecha'] >= '2025-03-17'].copy()

# hacemos la agrupación para el gráfico
df_pareto_volu = df_filtrado_pro.groupby('OS-Proveedor').agg({
    'OS-Cantidad': 'sum',
    'Valor total OS': 'sum'
}).reset_index()

# Ordenamos la cantidad de mayor a menor
df_pareto_volu = df_pareto_volu.sort_values(by='OS-Cantidad', ascending=False)

# Cálculo de acumulados sobre el VOLUMEN de producción
total_prendas = df_pareto_volu['OS-Cantidad'].sum()
df_pareto_volu['Porcentaje'] = (df_pareto_volu['OS-Cantidad'] / total_prendas) * 100
df_pareto_volu['Acumulado'] = df_pareto_volu['Porcentaje'].cumsum()

fig_vol = go.Figure()

# Barras de la cantidad de pedidos por cliente

fig_vol.add_trace(go.Bar(
    x=df_pareto_volu['OS-Proveedor'],
    y=df_pareto_volu['OS-Cantidad'],
    name='Volumen de Producción (Prendas)',
    marker_color="#632E8B", 
    # Pasamos datos extra al hover/etiqueta
    customdata=df_pareto_volu[['Valor total OS']], 
    hovertemplate=( 
        "<b>Proveedor: %{x}</b><br>" +
        "Cantidad de Prendas: %{y:,.0f}<br><br>" +
        "Valor total de OS: %{customdata[0]:$,.0f}<br>"
        "<extra></extra>"
    )
))

# Línea del % acumulado
fig_vol.add_trace(go.Scatter(
    x=df_pareto_volu['OS-Proveedor'],
    y=df_pareto_volu['Acumulado'],
    name='% Acumulado Volumen',
    yaxis='y2',
    line=dict(color="#4DEBBE", width=3),
    hovertemplate="Impacto Acumulado: %{y:.2f}%<extra></extra>"
))

# 6. Layout
fig_vol.update_layout(
    xaxis=dict(title='Proveedor', tickangle=-45),
    yaxis=dict(title='Cantidad de Prendas Producidas'),
    yaxis2=dict(title='Porcentaje Acumulado (%)', overlaying='y', side='right', range=[0, 105]),
    hovermode='x unified',
    template='plotly_white',
    height=700
)

# Línea del 80% 
fig_vol.add_shape(type="line", x0=0, x1=1, y0=80, y1=80, yref='y2', xref='paper',
                       line=dict(color="black", width=2, dash="dot"))

fig_vol.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=10,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_vol.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_vol.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_vol.show()

In [11]:
df_filtrado_prov = df_costos_fechacom[df_costos_fechacom['Fecha'] >= '2025-03-17'].copy()

# Agrupar por CLIENTE para el Pareto
df_pareto_valorpro = df_filtrado_prov.groupby('OS-Proveedor').agg({
    'OS-Cantidad': 'sum',
    'Valor total OS': 'sum'
}).reset_index()

# Ordenar por VALOR DE OP 
df_pareto_valorpro = df_pareto_valorpro.sort_values(by='Valor total OS', ascending=False)


total_valor_general = df_pareto_valorpro['Valor total OS'].sum()
df_pareto_valorpro['Porcentaje'] = (df_pareto_valorpro['Valor total OS'] / total_valor_general) * 100
df_pareto_valorpro['Acumulado'] = df_pareto_valorpro['Porcentaje'].cumsum()


fig_valor_pro = go.Figure()

# Barras: Valor Total OP por cliente
fig_valor_pro.add_trace(go.Bar(
    x=df_pareto_valorpro['OS-Proveedor'],
    y=df_pareto_valorpro['Valor total OS'],
    name='Valor Total OS ($)',
    marker_color='#632E8B', 
    customdata=df_pareto_valorpro[['OS-Cantidad']], 
    hovertemplate=(
        "<b>Proveedor: %{x}</b><br>" +
        "Valor Total OS: %{y:$,.0f}<br>" +
        "Total Prendas: %{customdata[0]:,.0f}<br>" + # <--- Cambio aquí: :.0f elimina decimales
        "<extra></extra>"
    )
))

# Línea: % Acumulado
fig_valor_pro.add_trace(go.Scatter(
    x=df_pareto_valorpro['OS-Proveedor'],
    y=df_pareto_valorpro['Acumulado'],
    name='% Acumulado',
    yaxis='y2',
    line=dict(color="#4DEBBE", width=3),
    hovertemplate="Impacto Acumulado: %{y:.2f}%<extra></extra>"
))

fig_valor_pro.update_layout(
    xaxis=dict(title='Proveedor', tickangle=-45),
    yaxis=dict(title='Valor Total OS ($)'),
    yaxis2=dict(title='Porcentaje Acumulado (%)', overlaying='y', side='right', range=[0, 105]),
    hovermode='x unified',
    template='plotly_white',
    height=700
)

# Línea del 80%
fig_valor_pro.add_shape(type="line", x0=0, x1=1, y0=80, y1=80, yref='y2', xref='paper',
                       line=dict(color="black", width=2, dash="dot"))

fig_valor_pro.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=10,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_valor_pro.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_valor_pro.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_valor_pro.show()

### Pareto de Cliente

In [12]:
# Filtro de fecha para tomar exactamente un año 
df_filtrado_cli = df_costos_valorop[df_costos_valorop['Fecha'] >= '2025-03-17'].copy()

# Agrupamos por el cliente y el número de la OP para sacar el valor exacto de la OP
df_ops_unicass = df_filtrado_cli.groupby(['Cliente', 'Num-OP']).agg({
    'Total Precio OP': 'mean', # Sacamos solo el promedio ya que el mismo valor totaal de la op se repite en todas las filas que tienen la misma op
    'OS-Cantidad': 'sum',
    'Valor total OS': 'sum'
}).reset_index()

# hacemos la agrupación final para el gráfico
df_pareto_vol = df_ops_unicass.groupby('Cliente').agg({
    'Total Precio OP': 'sum',
    'OS-Cantidad': 'sum',
    'Valor total OS': 'sum'
}).reset_index()

# Ordenamos la cantidad de mayor a menor
df_pareto_vol = df_pareto_vol.sort_values(by='OS-Cantidad', ascending=False)

# Cálculo de acumulados sobre el VOLUMEN de producción
total_prendas = df_pareto_vol['OS-Cantidad'].sum()
df_pareto_vol['Porcentaje'] = (df_pareto_vol['OS-Cantidad'] / total_prendas) * 100
df_pareto_vol['Acumulado'] = df_pareto_vol['Porcentaje'].cumsum()

fig_volumen = go.Figure()

# Barras de la cantidad de pedidos por cliente

fig_volumen.add_trace(go.Bar(
    x=df_pareto_vol['Cliente'],
    y=df_pareto_vol['OS-Cantidad'],
    name='Volumen de Producción (Prendas)',
    marker_color="#632E8B", 
    # Pasamos datos extra al hover/etiqueta
    customdata=df_pareto_vol[['Valor total OS','Total Precio OP']], 
    hovertemplate=( 
        "<b>Cliente: %{x}</b><br>" +
        "Cantidad de Prendas: %{y:,.0f}<br><br>" +
        "Valor total de OS: %{customdata[0]:$,.0f}<br>" +
        "Valor total de OP: %{customdata[1]:$,.0f}<br>"
        "<extra></extra>"
    )
))

# Línea del % acumulado
fig_volumen.add_trace(go.Scatter(
    x=df_pareto_vol['Cliente'],
    y=df_pareto_vol['Acumulado'],
    name='% Acumulado Volumen',
    yaxis='y2',
    line=dict(color="#4DEBBE", width=3),
    hovertemplate="Impacto Acumulado: %{y:.2f}%<extra></extra>"
))

# 6. Layout
fig_volumen.update_layout(
    xaxis=dict(title='Cliente', tickangle=-45),
    yaxis=dict(title='Cantidad de Prendas Producidas'),
    yaxis2=dict(title='Porcentaje Acumulado (%)', overlaying='y', side='right', range=[0, 105]),
    hovermode='x unified',
    template='plotly_white',
    height=700
)

# Línea del 80% 
fig_volumen.add_shape(type="line", x0=0, x1=1, y0=80, y1=80, yref='y2', xref='paper',
                       line=dict(color="black", width=2, dash="dot"))

fig_volumen.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=10,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_volumen.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_volumen.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_volumen.show()


In [13]:
df_filtrado_cliente = df_costos_valorop[df_costos_valorop['Fecha'] >= '2025-03-17'].copy()

# Agrupar por OP para no duplicar el 'Total Precio OP'
df_ops_unicas = df_filtrado_cliente.groupby(['Cliente', 'Num-OP']).agg({
    'Total Precio OP': 'mean', 
    'OS-Cantidad': 'sum',
    'Valor total OS': 'sum'
}).reset_index()

# Agrupar por CLIENTE para el Pareto
df_pareto_valor = df_ops_unicas.groupby('Cliente').agg({
    'Total Precio OP': 'sum',   # Ahora sí sumamos los valores únicos de sus OPs
    'OS-Cantidad': 'sum',
    'Valor total OS': 'sum'
}).reset_index()

# Ordenar por VALOR DE OP 
df_pareto_valor = df_pareto_valor.sort_values(by='Total Precio OP', ascending=False)


total_valor_general = df_pareto_valor['Total Precio OP'].sum()
df_pareto_valor['Porcentaje'] = (df_pareto_valor['Total Precio OP'] / total_valor_general) * 100
df_pareto_valor['Acumulado'] = df_pareto_valor['Porcentaje'].cumsum()


fig_valor = go.Figure()

# Barras: Valor Total OP por cliente
fig_valor.add_trace(go.Bar(
    x=df_pareto_valor['Cliente'],
    y=df_pareto_valor['Total Precio OP'],
    name='Valor Total OP ($)',
    marker_color='#632E8B', 
    customdata=df_pareto_valor[['Valor total OS', 'OS-Cantidad']], 
    hovertemplate=(
        "<b>Cliente: %{x}</b><br>" +
        "Valor Total OP: %{y:$,.0f}<br>" +
        "Valor Total OS: %{customdata[0]:$,.0f}<br>" +
        "Total Prendas: %{customdata[1]:,.0f}<br>" + # <--- Cambio aquí: :.0f elimina decimales
        "<extra></extra>"
    )
))

# Línea: % Acumulado
fig_valor.add_trace(go.Scatter(
    x=df_pareto_valor['Cliente'],
    y=df_pareto_valor['Acumulado'],
    name='% Acumulado',
    yaxis='y2',
    line=dict(color="#4DEBBE", width=3),
    hovertemplate="Impacto Acumulado: %{y:.2f}%<extra></extra>"
))

fig_valor.update_layout(
    xaxis=dict(title='Cliente', tickangle=-45),
    yaxis=dict(title='Valor Total OP ($)'),
    yaxis2=dict(title='Porcentaje Acumulado (%)', overlaying='y', side='right', range=[0, 105]),
    hovermode='x unified',
    template='plotly_white',
    height=700
)

# Línea del 80%
fig_valor.add_shape(type="line", x0=0, x1=1, y0=80, y1=80, yref='y2', xref='paper',
                       line=dict(color="black", width=2, dash="dot"))

fig_valor.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=10,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_valor.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_valor.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_valor.show()


### Pareto categorías

In [14]:
df_filtrado_cat = df_costos_fechacom[df_costos_fechacom['Fecha'] >= '2025-03-17'].copy()

# Utilizamos los mismos cálculos de ahorro que ya habiamos hecho anteriormente (Diferencia_Unitaria y Ahorro_Potencial_Total)

group_cols = ['Categoría', 'Cliente', 'Proceso Producción']
df_filtrado_cat['min_val'] = df_filtrado_cat.groupby(group_cols)['OS-Valor Unitario'].transform('min')
df_filtrado_cat['Ahorro_Potencial_Total'] = (df_filtrado_cat['OS-Valor Unitario'] - df_filtrado_cat['min_val']) * df_filtrado_cat['OS-Cantidad']

# Agrupamos por categoría, sumamos el ahorro y traemos el conteo de procesos para tener contexto
df_pareto_cat = df_filtrado_cat.groupby('Categoría').agg({
    'Ahorro_Potencial_Total': 'sum',
    'Valor total OS': 'sum',
    'Proceso Producción': 'nunique' # Cuántos procesos distintos hay en esta categoría
}).reset_index()

# 4. Ordenar y Acumulados
df_pareto_cat = df_pareto_cat.sort_values(by='Ahorro_Potencial_Total', ascending=False)
total_ahorro_cat = df_pareto_cat['Ahorro_Potencial_Total'].sum()
df_pareto_cat['Acumulado'] = (df_pareto_cat['Ahorro_Potencial_Total'].cumsum() / total_ahorro_cat) * 100


fig_cat_ahorro = go.Figure()

fig_cat_ahorro.add_trace(go.Bar(
    x=df_pareto_cat['Categoría'],
    y=df_pareto_cat['Ahorro_Potencial_Total'],
    name='Oportunidad de Ahorro ($)',
    marker_color='#3366CC',
    customdata=df_pareto_cat[['Valor total OS', 'Proceso Producción']],
    hovertemplate=(
        "<b>Categoría: %{x}</b><br>" +
        "Ahorro potencial total: %{y:$,.0f}<br>" +
        "Valor total OS: %{customdata[0]:$,.0f}<br>" +
        "Variedad de Procesos: %{customdata[1]}<extra></extra>"
    )
))

fig_cat_ahorro.add_trace(go.Scatter(
    x=df_pareto_cat['Categoría'],
    y=df_pareto_cat['Acumulado'],
    name='% Acumulado',
    yaxis='y2',
    line=dict(color='#FF9900', width=3),
    hovertemplate="Impacto Acumulado: %{y:.2f}%<extra></extra>"
))

# Layout
fig_cat_ahorro.update_layout(
    xaxis=dict(title='Categoría'),
    yaxis=dict(title='Ahorro Potencial Total ($)'),
    yaxis2=dict(title='Porcentaje Acumulado (%)', overlaying='y', side='right', range=[0, 105]),
    hovermode='x unified',
    template='plotly_white'
)

# Línea del 80%
fig_cat_ahorro.add_shape(type="line", x0=0, x1=1, y0=80, y1=80, yref='y2', xref='paper',
                  line=dict(color="black", width=2, dash="dot"))

fig_cat_ahorro.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=10,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_cat_ahorro.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_cat_ahorro.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_cat_ahorro.show()

In [16]:
# Filtro de fecha para tomar exactamente un año 
df_filtrado_cate = df_costos_fechacom[df_costos_fechacom['Fecha'] >= '2025-03-17'].copy()

# hacemos la agrupación final para el gráfico
df_pareto_vol_cat = df_filtrado_cate.groupby('Categoría').agg({
    'OS-Cantidad': 'sum',
    'Valor total OS': 'sum'
}).reset_index()

# Ordenamos la cantidad de mayor a menor
df_pareto_vol_cat = df_pareto_vol_cat.sort_values(by='OS-Cantidad', ascending=False)

# Cálculo de acumulados sobre el VOLUMEN de producción
total_prendas = df_pareto_vol_cat['OS-Cantidad'].sum()
df_pareto_vol_cat['Porcentaje'] = (df_pareto_vol_cat['OS-Cantidad'] / total_prendas) * 100
df_pareto_vol_cat['Acumulado'] = df_pareto_vol_cat['Porcentaje'].cumsum()

fig_volumenc = go.Figure()

# Barras de la cantidad de pedidos por cliente

fig_volumenc.add_trace(go.Bar(
    x=df_pareto_vol_cat['Categoría'],
    y=df_pareto_vol_cat['OS-Cantidad'],
    name='Volumen de Producción (Prendas)',
    marker_color="#632E8B", 
    # Pasamos datos extra al hover/etiqueta
    customdata=df_pareto_vol_cat[['Valor total OS']], 
    hovertemplate=( 
        "<b>Cliente: %{x}</b><br>" +
        "Cantidad de Prendas: %{y:,.0f}<br><br>" +
        "Valor total de OS: %{customdata[0]:$,.0f}<br>" +
        "<extra></extra>"
    )
))

# Línea del % acumulado
fig_volumenc.add_trace(go.Scatter(
    x=df_pareto_vol_cat['Categoría'],
    y=df_pareto_vol_cat['Acumulado'],
    name='% Acumulado Volumen',
    yaxis='y2',
    line=dict(color="#4DEBBE", width=3),
    hovertemplate="Impacto Acumulado: %{y:.2f}%<extra></extra>"
))

# 6. Layout
fig_volumenc.update_layout(
    xaxis=dict(title='Cat', tickangle=-45),
    yaxis=dict(title='Cantidad de Prendas Producidas'),
    yaxis2=dict(title='Porcentaje Acumulado (%)', overlaying='y', side='right', range=[0, 105]),
    hovermode='x unified',
    template='plotly_white',
    height=700
)

# Línea del 80% 
fig_volumenc.add_shape(type="line", x0=0, x1=1, y0=80, y1=80, yref='y2', xref='paper',
                       line=dict(color="black", width=2, dash="dot"))

fig_volumenc.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=10,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_volumenc.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_volumenc.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_volumenc.show()


## Variación de costos que manejan los proveedores en las mismas categorías para diferentes clientes

In [11]:
top_10_proveedores = df_costos_fechacom.groupby('OS-Proveedor')['Valor total OS'].sum().nlargest(10).index

# Filtramos por el top 10 de categorías y proveedores
df_prov_cli = df_costos_fechacom[
    (df_costos_fechacom['Categoría'].isin(top_10_cats)) & 
    (df_costos_fechacom['OS-Proveedor'].isin(top_10_proveedores))
].copy()

# Agrupamos incluyendo al CLIENTE
df_plot_prov_cli = df_prov_cli.groupby(['OS-Proveedor', 'Categoría', 'Cliente']).agg({
    'OS-Valor Unitario': 'mean',
    'OS-Cantidad': 'sum',
    'Valor total OS': 'sum'
}).reset_index()

lista_clientes = sorted(df_plot_prov_cli['Cliente'].unique())
lista_proveedores = sorted(df_plot_prov_cli['OS-Proveedor'].unique())

fig_prov_cli = go.Figure()

# 3. Creamos los rastros (Traces)
# Necesitamos crear un rastro por cada PROVEEDOR para que el barmode='group' funcione
# Pero lo haremos dinámico para que el filtro de cliente cambie los datos
for prov in lista_proveedores:
    df_temp = df_plot_prov_cli[df_plot_prov_cli['OS-Proveedor'] == prov]
    
    # Iniciamos mostrando el promedio General (Todos los clientes)
    # Para eso, recalculamos el promedio por categoría ignorando el cliente inicialmente
    df_gen = df_temp.groupby('Categoría').agg({'OS-Valor Unitario': 'mean'}).reset_index()
    
    fig_prov_cli.add_trace(go.Bar(
        x=df_gen['Categoría'],
        y=df_gen['OS-Valor Unitario'],
        name=prov,
        visible=True
    ))


botones_cliente = []
botones_cliente.append(dict(
    label="Todos los Clientes",
    method="update",
    args=[{"y": [df_plot_prov_cli[df_plot_prov_cli['OS-Proveedor'] == p].groupby('Categoría')['OS-Valor Unitario'].mean().values for p in lista_proveedores],
           "x": [df_plot_prov_cli[df_plot_prov_cli['OS-Proveedor'] == p].groupby('Categoría')['Categoría'].unique().tolist() for p in lista_proveedores]},
          {"title": "Costo Promedio por Proveedor: Todos los Clientes"}]
))

# Opciones por cada Cliente específico
for cli in lista_clientes:
    # Preparamos los datos Y para cada proveedor basándonos en ese cliente
    y_por_proveedor = []
    x_por_proveedor = []
    
    for prov in lista_proveedores:
        df_sub = df_plot_prov_cli[(df_plot_prov_cli['Cliente'] == cli) & (df_plot_prov_cli['OS-Proveedor'] == prov)]
        y_por_proveedor.append(df_sub['OS-Valor Unitario'].values)
        x_por_proveedor.append(df_sub['Categoría'].values)

    botones_cliente.append(dict(
        label=cli,
        method="update",
        args=[{"y": y_por_proveedor, "x": x_por_proveedor},
              {"title": f"Costo por Proveedor para: {cli}"}]
    ))

fig_prov_cli.update_layout(
    updatemenus=[dict(
        buttons=botones_cliente,
        direction="down",
        showactive=True,
        x=0, y=1.2
    )],
    barmode='group',
    xaxis_title="Categoría",
    yaxis_title="Costo Unitario Promedio ($)",
    template='plotly_white',
    height=650,
    legend_title_text='Proveedor'
)

fig_prov_cli.show()

## Gráfico impacto total por servicio

In [12]:
df_impacto = df_costos.copy()

df_impacto['Sobrecosteo'] = df_impacto['Diferencia'].apply(lambda x: abs(x) if x < 0 else 0)
df_impacto['Subcosteo'] = df_impacto['Diferencia'].apply(lambda x: x if x > 0 else 0)

df_resumen = df_impacto.groupby('Proceso Producción').agg({
    'Sobrecosteo': 'sum',
    'Subcosteo': 'sum',
    'Diferencia': 'sum'
}).reset_index()

df_resumen = df_resumen.rename(columns={'Diferencia': 'Balance'})

df_melt = df_resumen.melt(
    id_vars='Proceso Producción',
    value_vars=['Sobrecosteo', 'Subcosteo', 'Balance'],
    var_name='tipo',
    value_name='valor'
)

figimpactoservicio = px.bar(
    df_melt,
    x='Proceso Producción',
    y='valor',
    color='tipo',
    barmode='group',
    color_discrete_map={
        'Sobrecosteo': '#2E7D32',      # verde elegante
        'Subcosteo': '#C62828',  # rojo fuerte
        'Balance': '#424242'      # gris neutro
    }
)

# Agregar línea de curva acumulativa para Balance
import plotly.graph_objects as go

df_balance = df_melt[df_melt['tipo'] == 'Balance'].copy()
procesos_orden = df_melt['Proceso Producción'].unique()
df_balance = df_balance.set_index('Proceso Producción').reindex(procesos_orden).reset_index()
df_balance['cumsum'] = df_balance['valor'].cumsum()

# Calcular porcentajes
total_balance = df_balance['cumsum'].iloc[-1]
df_balance['porcentaje'] = (df_balance['cumsum'] / total_balance * 100).round(1)

figimpactoservicio.add_trace(go.Scatter(
    x=df_balance['Proceso Producción'],
    y=df_balance['cumsum'],
    mode='lines+markers+text',
    name='Curva Acumulativa Balance',
    line=dict(color='#424242', width=2),
    text=df_balance['porcentaje'].astype(str) + '%',
    textposition='top left',
    textfont=dict(color='#424242', size=10)
))

df_ahorro = df_melt[df_melt['tipo'] == 'Sobrecosteo'].copy()
procesos_orden_ahorro = df_melt['Proceso Producción'].unique()
df_ahorro = df_ahorro.set_index('Proceso Producción').reindex(procesos_orden_ahorro).reset_index()
df_ahorro['cumsum'] = df_ahorro['valor'].cumsum()

# Calcular porcentajes para Sobrecosteo
total_ahorro = df_ahorro['cumsum'].iloc[-1]
df_ahorro['porcentaje'] = (df_ahorro['cumsum'] / total_ahorro * 100).round(1)

figimpactoservicio.add_trace(go.Scatter(
    x=df_ahorro['Proceso Producción'],
    y=df_ahorro['cumsum'],
    mode='lines+markers+text',
    name='Curva Acumulativa Sobrecosteo',
    line=dict(color='#2E7D32', width=2),
    text=df_ahorro['porcentaje'].astype(str) + '%',
    textposition='bottom right',
    textfont=dict(color='#2E7D32', size=10)
))

df_sobrecosto = df_melt[df_melt['tipo'] == 'Subcosteo'].copy()
procesos_orden_sobrecosto = df_melt['Proceso Producción'].unique()
df_sobrecosto = df_sobrecosto.set_index('Proceso Producción').reindex(procesos_orden_sobrecosto).reset_index()
df_sobrecosto['cumsum'] = df_sobrecosto['valor'].cumsum()

# Calcular porcentajes para Subcosteo
total_sobrecosto = df_sobrecosto['cumsum'].iloc[-1]
df_sobrecosto['porcentaje'] = (df_sobrecosto['cumsum'] / total_sobrecosto * 100).round(1)

figimpactoservicio.add_trace(go.Scatter(
    x=df_sobrecosto['Proceso Producción'],
    y=df_sobrecosto['cumsum'],
    mode='lines+markers+text',
    name='Curva Acumulativa Subcosteo',
    line=dict(color='#C62828', width=2),
    text=df_sobrecosto['porcentaje'].astype(str) + '%',
    textposition='bottom left',
    textfont=dict(color='#C62828', size=10)
))

figimpactoservicio.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
figimpactoservicio.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

figimpactoservicio.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

figimpactoservicio.show()

## BOXPLOT de la diferencia de costos en cada proceso de producción

In [13]:
# Creamos un gráfico de caja para visualizar la distribución de la diferencia del valor unitario por proceso de producción
df_costos.columns = df_costos.columns.str.strip()

df_costos['Proceso Producción'] = pd.Categorical(
    df_costos['Proceso Producción'],
    categories=sorted(df_costos['Proceso Producción'].unique()),
    ordered=True
)

figboxplot = px.box(df_costos, 
             x='Proceso Producción', 
             y='Diferencia valor unitario', 
             points='outliers',
             notched=True,
             color='Proceso Producción',
             hover_data = ['OP Det', 'Nombre del Comercial', 'Costeo_Producto'],
             color_discrete_sequence=paleta_textil,
             category_orders={'Proceso Producción': sorted(df_costos['Proceso Producción'].unique())})

figboxplot.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
figboxplot.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

figboxplot.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

figboxplot.show()

In [14]:
print(df_costos['Proceso Producción'].dtype)
print(df_costos['Proceso Producción'].unique())

category
['Bordado', 'Paquete Completo', 'Confección', 'Estampado', 'Prelavado', 'Sublimación', 'Paquete Completo Sin Marca', 'Corte y Confeccion']
Categories (8, object): ['Bordado' < 'Confección' < 'Corte y Confeccion' < 'Estampado' < 'Paquete Completo' < 'Paquete Completo Sin Marca' < 'Prelavado' < 'Sublimación']


## BOXPLOT Diferencia de valor total

In [15]:
df_costos.columns = df_costos.columns.str.strip()
figboxtotal = px.box(df_costos, 
             x='Proceso Producción', 
             y='Diferencia', 
             points='outliers',
             notched=True,
             color='Proceso Producción',
             hover_data = ['OP Det', 'Nombre del Comercial', 'Costeo_Producto', 'OS-Valor Unitario', 'costo_antes_iva' ],
             color_discrete_sequence=paleta_textil,
             category_orders={'Proceso Producción': sorted(df_costos['Proceso Producción'].unique())})

figboxtotal.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
figboxtotal.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

figboxtotal.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

figboxtotal.show()

## BOXPLOT coeficiente de variación en categorías por diferencia de valor unitario

In [16]:
# Utilizamos el promedio de la diferencia para identificar las 5 categorías con mayor impacto económico.
top5_categorias = (df_costos.groupby('Categoría')['Diferencia valor unitario']
                   .mean()
                   .sort_values(ascending=False)
                   .head(5)
                   .index.tolist()
                   )


df_filtrado = df_costos[df_costos['Categoría'].isin(top5_categorias)].copy()

df_filtrado['variacion_netadif'] = df_filtrado.groupby(['Categoría', 'Proceso Producción'])['Diferencia valor unitario'].transform(lambda x: x - x.mean())

# --- PASO 4: Gráfica de "Impacto Económico" ---
fig_dif = px.box(df_filtrado, 
             x='Categoría', 
             y='variacion_netadif', 
             color='Proceso Producción',
             points='outliers',
             hover_data = ['OP Det', 'Nombre del Comercial', 'Costeo_Producto', 'OS-Valor Unitario', 'costo_antes_iva' ],
             color_discrete_sequence=paleta_textil)

# Línea de referencia en cero
fig_dif.add_hline(y=0, line_dash="dash", line_color="blue")

fig_dif.update_layout(
    boxmode='group',
    yaxis_title="Desviación del Costo ($)",
    xaxis_title="Top 5 Categorías con Mayor Impacto Económico"
)

fig_dif.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_dif.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_dif.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_dif.show()

C:\Users\Laura\AppData\Local\Temp\ipykernel_23976\2237638566.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_filtrado['variacion_netadif'] = df_filtrado.groupby(['Categoría', 'Proceso Producción'])['Diferencia valor unitario'].transform(lambda x: x - x.mean())


## Impacto económico general

In [17]:
# Sumamos la diferencia por OP para que solo exista una barra por etiqueta
df_consolidado_total = df_costos.groupby(['Categoría', 'Proceso Producción']).agg({
    'Diferencia': 'sum'
}).reset_index()

# Ahora sí calculamos el impacto absoluto sobre el total de la OP
df_consolidado_total['abs_impacto'] = df_consolidado_total['Diferencia'].abs()

top_impacto_total = df_consolidado_total.sort_values(by='abs_impacto', ascending=False).head(20)

top_impacto_total = top_impacto_total.sort_values(by='Diferencia', ascending=False)

top_impacto_total['Etiqueta'] = top_impacto_total['Categoría'] + " (" + top_impacto_total['Proceso Producción'].astype(str) + ")"

# Agregar la columna estado basada en Diferencia
top_impacto_total['estado'] = top_impacto_total['Diferencia'].apply(
    lambda x: 'Ahorro' if x < 0 else 'Sobrecosto'
)

fig_total = px.bar(
    top_impacto_total,
    x='Diferencia',
    y='Etiqueta',
    orientation='h',
    color='estado',
    color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
)

# Esto elimina las líneas negras que dividen los segmentos dentro de la barra
fig_total.update_traces(marker_line_width=0) 
fig_total.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=12,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_total.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_total.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_total.show()

C:\Users\Laura\AppData\Local\Temp\ipykernel_23976\1593791139.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_consolidado_total = df_costos.groupby(['Categoría', 'Proceso Producción']).agg({


## Impacto económico por comercial

In [18]:
import numpy as np

df_heatmap = df_costos.pivot_table(
    index='Nombre del Comercial',
    columns='Proceso Producción',
    values='Diferencia',
    aggfunc='sum'  # puedes cambiar a sum si quieres impacto total
)
df_mediana = df_costos.pivot_table(
    index='Nombre del Comercial',
    columns='Proceso Producción',
    values='Diferencia valor unitario',
    aggfunc='median'
)

df_media = df_costos.pivot_table(
    index='Nombre del Comercial',
    columns='Proceso Producción',
    values='Diferencia valor unitario',
    aggfunc='mean'
)
df_media = df_media.reindex_like(df_heatmap).fillna(0)
df_mediana = df_mediana.reindex_like(df_heatmap).fillna(0)

customdata = np.dstack((
    df_media.values,
    df_mediana.values
))

figcomercial = px.imshow(
    df_heatmap,
    text_auto=True,
    color_continuous_scale='RdYlGn_r',  
    aspect='auto'
)

figcomercial.update_traces(
    customdata=customdata,
    hovertemplate=
        "Comercial: %{y}<br>" +
        "Proceso: %{x}<br><br>" +
        "Diferencia: %{z:,.0f}<br>" +
        #"Media: %{customdata[0]:,.0f}<br>" +
        "Mediana: %{customdata[1]:,.0f}<br>" +
        "<extra></extra>"
)

# Limitar escala a percentil 95 para mejor visualización de diferencias
percentil = np.percentile(np.abs(df_heatmap.values), 95)

figcomercial.update_layout(
    # Centrar escala de color en 0 (usando percentil para evitar valores extremos)
    coloraxis=dict(
        cmin=-1000000,
        cmax=1000000,
        cmid=0
    ),
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
figcomercial.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

figcomercial.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

figcomercial.show()

C:\Users\Laura\AppData\Local\Temp\ipykernel_23976\1552998528.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df_heatmap = df_costos.pivot_table(
C:\Users\Laura\AppData\Local\Temp\ipykernel_23976\1552998528.py:9: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df_mediana = df_costos.pivot_table(
C:\Users\Laura\AppData\Local\Temp\ipykernel_23976\1552998528.py:16: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df_media = df_costos.pivot_table(


In [19]:
df_heatmap.values

array([[ 7.2000000e+03, -4.0000000e+03,  0.0000000e+00, -5.5000000e+04,
         0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
       [-2.9200000e+04, -2.8535460e+07,  0.0000000e+00,  4.7040000e+05,
         0.0000000e+00,  2.8000000e+05,  0.0000000e+00, -1.1912600e+07],
       [ 6.1970000e+05, -5.0280000e+06,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00,  0.0000000e+00, -1.4384000e+06],
       [ 6.0000000e+04, -5.8000000e+04,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
       [ 1.4897000e+06,  2.4219901e+06,  1.0500000e+05, -6.3600000e+04,
         0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
       [-9.0000000e+04, -6.0000000e+05,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
       [ 6.7200000e+04, -5.1400000e+05,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00,  1.3600000e+04,  0

## Valor costeado vs pagado en margen de ahorro o sobrecosto

### Confección

In [20]:
df_confeccion = df_costos[df_costos['Proceso Producción'] == 'Confección'].copy()

# Sumamos la diferencia por OP para que solo exista una barra por etiqueta
df_consolidado = df_confeccion.groupby(['Categoría', 'OP Det']).agg({
    'Diferencia': 'sum'
}).reset_index()

# Ahora sí calculamos el impacto absoluto sobre el total de la OP
df_consolidado['abs_impacto'] = df_consolidado['Diferencia'].abs()

# Top sobrecosto (mayor impacto positivo)
top_sobrecosto = df_consolidado.nlargest(10, 'Diferencia')
top_ahorro = df_consolidado.nsmallest(10, 'Diferencia')

# Unir ambos
top_impacto = pd.concat([top_sobrecosto, top_ahorro], ignore_index=True)

# Ordenar de mayor a menor Diferencia (rojo arriba, verde abajo)
top_impacto = top_impacto.sort_values(by='Diferencia', ascending=False)

top_impacto['Etiqueta'] = top_impacto['Categoría'] + " (OP: " + top_impacto['OP Det'].astype(str) + ")"

top_impacto['estado'] = top_impacto['Diferencia'].apply(
    lambda x: 'Ahorro' if x < 0 else 'Sobrecosto'
)

fig = px.bar(
    top_impacto,
    x='Diferencia',
    y='Etiqueta',
    orientation='h',
    color='estado',
    color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
)

# Esto elimina las líneas negras que dividen los segmentos dentro de la barra
fig.update_traces(marker_line_width=0) 
fig.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig.show()

In [21]:
# Filtramos el DF para obtener solo los datos de confección.
df_conf = df_costos[df_costos['Proceso Producción'] == 'Confección'].copy()

# Definimos una columna de estado para tener el contexto de qué significa cada punto en la gráfica.
df_conf['estado'] = df_conf.apply(
    lambda 
    x: 'Ahorro' 
    if x['OS-Valor Unitario'] < x['costo_antes_iva'] 
    else 'Sobrecosto',
    axis=1
)

# Creamos la gráfica de confección, lo que esta encima de la línea diagonal es ahorro y lo que esta debajo es sobrecosto
# El tamaño de cada punto representa el impacto económico

fig_confe = px.scatter(
    df_conf,
    x='OS-Valor Unitario',
    y='costo_antes_iva',
    color='Categoría',
    color_discrete_sequence=paleta_textil,
    hover_data=['OP Det', 'Nombre del Comercial', 'Costeo_Producto', 'OS-Valor Unitario', 'costo_antes_iva'],
    title='Confección'
)

min_val = min(df_conf['costo_antes_iva'].min(), df_conf['OS-Valor Unitario'].min())
max_val = max(df_conf['costo_antes_iva'].max(), df_conf['OS-Valor Unitario'].max())

fig_confe.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="pink", dash="dash")
)
fig_confe.update_traces(marker=dict(size=5)) 

fig_confe.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_confe.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1,
    title='Costo OS antes IVA'
)

fig_confe.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray',
    title='Costo antes IVA (costeado)'
)
fig_confe.show()


### Sublimación

In [22]:
# Filtramos el DF para obtener solo los datos de sublimación.
df_subl = df_costos[df_costos['Proceso Producción'] == 'Sublimación'].copy()

# Definimos una columna de estado para tener el contexto de qué significa cada punto en la gráfica.
df_subl['estado'] = df_subl.apply(
    lambda 
    x: 'Ahorro' 
    if x['OS-Valor Unitario'] < x['costo_antes_iva'] 
    else 'Sobrecosto',
    axis=1
)


fig_subl= px.scatter(
    df_subl,
    x='OS-Valor Unitario',
    y='costo_antes_iva',
    color='OP Det',
    color_discrete_sequence=paleta_textil,
    hover_data=['OP Det', 'Nombre del Comercial', 'Costeo_Producto', 'OS-Valor Unitario', 'costo_antes_iva'],
    title='Sublimación'
)

min_val = min(df_subl['costo_antes_iva'].min(), df_subl['OS-Valor Unitario'].min())
max_val = max(df_subl['costo_antes_iva'].max(), df_subl['OS-Valor Unitario'].max())

fig_subl.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="pink", dash="dash")
)
fig_subl.update_traces(marker=dict(size=15)) 

fig_subl.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_subl.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1,
    title='Costo OS antes IVA'
)

fig_subl.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray',
    title='Costo antes IVA (costeado)'
)

fig_subl.show()

In [23]:
df_subli = df_costos[df_costos['Proceso Producción'] == 'Sublimación']

# Sumamos la diferencia por OP para que solo exista una barra por etiqueta
df_consolidado_subl = df_subli.groupby(['Categoría', 'OP Det']).agg({
    'Diferencia': 'sum'
}).reset_index()

# Ahora sí calculamos el impacto absoluto sobre el total de la OP
df_consolidado_subl['abs_impacto'] = df_consolidado_subl['Diferencia'].abs()

# Top de las OPs con mayor impacto real
top_sobrecosto_sub = df_consolidado_subl.nlargest(7, 'Diferencia')
top_ahorro_sub = df_consolidado_subl.nsmallest(7, 'Diferencia')

# Unir ambos
top_impacto_sub = pd.concat([top_sobrecosto_sub, top_ahorro_sub], ignore_index=True)

# Ordenar de mayor a menor Diferencia (rojo arriba, verde abajo)
top_impacto_sub = top_impacto_sub.sort_values(by='Diferencia', ascending=False)

top_impacto_sub['Etiqueta'] = top_impacto_sub['Categoría'] + " (OP: " + top_impacto_sub['OP Det'].astype(str) + ")"

top_impacto_sub['estado'] = top_impacto_sub['Diferencia'].apply(
    lambda x: 'Ahorro' if x < 0 else 'Sobrecosto'
)

fig_sub = px.bar(
    top_impacto_sub,
    x='Diferencia',
    y='Etiqueta',
    orientation='h',
    color='estado',
    color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    template='plotly_dark'
)

# Esto elimina las líneas negras que dividen los segmentos dentro de la barra
fig_sub.update_traces(marker_line_width=0) 
fig_sub.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_sub.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_sub.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_sub.show()

### Bordado

In [24]:
# Filtramos el DF para obtener solo los datos de Bordado.
df_bord = df_costos[df_costos['Proceso Producción'] == 'Bordado'].copy()

# Definimos una columna de estado para tener el contexto de qué significa cada punto en la gráfica.
df_bord['estado'] = df_bord.apply(
    lambda 
    x: 'Ahorro' 
    if x['OS-Valor Unitario'] < x['costo_antes_iva'] 
    else 'Sobrecosto',
    axis=1
)

# Creamos la gráfica de bordado, lo que esta encima de la línea diagonal es ahorro y lo que esta debajo es sobrecosto
# El tamaño de cada punto representa el impacto económico

fig_bordado= px.scatter(
    df_bord,
    x='OS-Valor Unitario',
    y='costo_antes_iva',
    color='Categoría',
    color_discrete_sequence=paleta_textil,
    hover_data=['OP Det', 'Nombre del Comercial', 'Costeo_Producto', 'OS-Valor Unitario', 'costo_antes_iva'],
    title='Bordado',
)

min_val = min(df_bord['costo_antes_iva'].min(), df_bord['OS-Valor Unitario'].min())
max_val = max(df_bord['costo_antes_iva'].max(), df_bord['OS-Valor Unitario'].max())

fig_bordado.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="pink", dash="dash")
)

fig_bordado.update_traces(marker=dict(size=5)) 

fig_bordado.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_bordado.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1,
    title='Costo OS antes IVA'
)

fig_bordado.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray',
    title='Costo antes IVA (costeado)'
)
fig_bordado.show()

In [25]:
df_bord = df_costos[df_costos['Proceso Producción'] == 'Bordado']

# Sumamos la diferencia por OP para que solo exista una barra por etiqueta
df_consolidado_bord = df_bord.groupby(['Categoría', 'OP Det']).agg({
    'Diferencia': 'sum'
}).reset_index()

# Ahora sí calculamos el impacto absoluto sobre el total de la OP
df_consolidado_bord['abs_impacto'] = df_consolidado_bord['Diferencia'].abs()

top_sobrecosto_bord = df_consolidado_bord.nlargest(10, 'Diferencia')
top_ahorro_bord = df_consolidado_bord.nsmallest(10, 'Diferencia')

# Unir ambos
top_impacto_bord = pd.concat([top_sobrecosto_bord, top_ahorro_bord], ignore_index=True)

# Ordenar de mayor a menor Diferencia (rojo arriba, verde abajo)
top_impacto_bord = top_impacto_bord.sort_values(by='Diferencia', ascending=False)

top_impacto_bord['Etiqueta'] = top_impacto_bord['Categoría'] + " (OP: " + top_impacto_bord['OP Det'].astype(str) + ")"

top_impacto_bord['estado'] = top_impacto_bord['Diferencia'].apply(
    lambda x: 'Ahorro' if x < 0 else 'Sobrecosto'
)

fig_bord = px.bar(
    top_impacto_bord,
    x='Diferencia',
    y='Etiqueta',
    orientation='h',
    color='estado',
    color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    template='plotly_dark'
)

# Esto elimina las líneas negras que dividen los segmentos dentro de la barra
fig_bord.update_traces(marker_line_width=0) 

fig_bord.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_bord.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_bord.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_bord.show()

### Estampado

In [26]:
# Filtramos el DF para obtener solo los datos de Bordado.
df_est = df_costos[df_costos['Proceso Producción'] == 'Estampado'].copy()

# Definimos una columna de estado para tener el contexto de qué significa cada punto en la gráfica.
df_est['estado'] = df_est.apply(
    lambda 
    x: 'Ahorro' 
    if x['OS-Valor Unitario'] < x['costo_antes_iva'] 
    else 'Sobrecosto',
    axis=1
)

# Creamos la gráfica de bordado, lo que esta encima de la línea diagonal es ahorro y lo que esta debajo es sobrecosto
# El tamaño de cada punto representa el impacto económico

fig_estampado= px.scatter(
    df_est,
    x='OS-Valor Unitario',
    y='costo_antes_iva',
    color='Categoría',
    color_discrete_sequence=paleta_textil,
    hover_data=['OP Det', 'Nombre del Comercial', 'Costeo_Producto', 'OS-Valor Unitario', 'costo_antes_iva'],
    title='Estampado',
)

min_val = min(df_est['costo_antes_iva'].min(), df_est['OS-Valor Unitario'].min())
max_val = max(df_est['costo_antes_iva'].max(), df_est['OS-Valor Unitario'].max())

fig_estampado.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="pink", dash="dash")
)

fig_estampado.update_traces(marker=dict(size=5)) 

fig_estampado.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_estampado.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1,
    title='Costo OS antes IVA'
)

fig_estampado.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray',
    title='Costo antes IVA (costeado)'
)
fig_estampado.show()

In [27]:
df_est = df_costos[df_costos['Proceso Producción'] == 'Estampado']

# Sumamos la diferencia por OP para que solo exista una barra por etiqueta
df_consolidado_est = df_est.groupby(['Categoría', 'OP Det']).agg({
    'Diferencia': 'sum'
}).reset_index()

# Ahora sí calculamos el impacto absoluto sobre el total de la OP
df_consolidado_est['abs_impacto'] = df_consolidado_est['Diferencia'].abs()

top_sobrecosto_est = df_consolidado_est.nlargest(10, 'Diferencia')
top_ahorro_est = df_consolidado_est.nsmallest(10, 'Diferencia')

# Unir ambos
top_impacto_est = pd.concat([top_sobrecosto_est, top_ahorro_est], ignore_index=True)

# Ordenar de mayor a menor Diferencia (rojo arriba, verde abajo)
top_impacto_est = top_impacto_est.sort_values(by='Diferencia', ascending=False)

top_impacto_est['Etiqueta'] = top_impacto_est['Categoría'] + " (OP: " + top_impacto_est['OP Det'].astype(str) + ")"

top_impacto_est['estado'] = top_impacto_est['Diferencia'].apply(
    lambda x: 'Ahorro' if x < 0 else 'Sobrecosto'
)

fig_est = px.bar(
    top_impacto_est,
    x='Diferencia',
    y='Etiqueta',
    orientation='h',
    color='estado',
    color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    template='plotly_dark'
)

# Esto elimina las líneas negras que dividen los segmentos dentro de la barra
fig_est.update_traces(marker_line_width=0) 

fig_est.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_est.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_est.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_est.show()

### Paquete Completo Sin Marca

In [28]:
# Filtramos el DF para obtener solo los datos de Bordado.
df_pqt_sin = df_costos[df_costos['Proceso Producción'] == 'Paquete Completo Sin Marca'].copy()

# Definimos una columna de estado para tener el contexto de qué significa cada punto en la gráfica.
df_pqt_sin['estado'] = df_pqt_sin.apply(
    lambda 
    x: 'Ahorro' 
    if x['OS-Valor Unitario'] < x['costo_antes_iva'] 
    else 'Sobrecosto',
    axis=1
)

# Creamos la gráfica de bordado, lo que esta encima de la línea diagonal es ahorro y lo que esta debajo es sobrecosto
# El tamaño de cada punto representa el impacto económico

fig_pqt_sin= px.scatter(
    df_pqt_sin,
    x='OS-Valor Unitario',
    y='costo_antes_iva',
    color='Categoría',
    color_discrete_sequence=paleta_textil,
    hover_data=['OP Det', 'Nombre del Comercial', 'Costeo_Producto', 'OS-Valor Unitario', 'costo_antes_iva'],
    title='Paquete Completo Sin Marca',
)

min_val = min(df_pqt_sin['costo_antes_iva'].min(), df_pqt_sin['OS-Valor Unitario'].min())
max_val = max(df_pqt_sin['costo_antes_iva'].max(), df_pqt_sin['OS-Valor Unitario'].max())

fig_pqt_sin.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="pink", dash="dash")
)

fig_pqt_sin.update_traces(marker=dict(size=15)) 

fig_pqt_sin.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_pqt_sin.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1,
    title='Costo OS antes IVA'
)

fig_pqt_sin.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray',
    title='Costo antes IVA (costeado)'
)
fig_pqt_sin.show()

In [29]:
# Solo creamos la columna de estado para el color del gráfico
df_costos['estado'] = df_costos['Diferencia'].apply(
    lambda x: 'Ahorro' if x < 0 else 'Sobrecosto'
)

df_pqt_sin = df_costos[df_costos['Proceso Producción'] == 'Paquete Completo Sin Marca']

# Sumamos la diferencia por OP para que solo exista una barra por etiqueta
df_consolidado_pqsin = df_pqt_sin.groupby(['Categoría', 'OP Det']).agg({
    'Diferencia': 'sum'
}).reset_index()

# Ahora sí calculamos el impacto absoluto sobre el total de la OP
df_consolidado_pqsin['abs_impacto'] = df_consolidado_pqsin['Diferencia'].abs()

top_sobrecosto_pqtsin = df_consolidado_pqsin.nlargest(10, 'Diferencia')
top_ahorro_pqtsin= df_consolidado_pqsin.nsmallest(10, 'Diferencia')

# Unir ambos
top_impacto_pqsin = pd.concat([top_sobrecosto_pqtsin, top_ahorro_pqtsin], ignore_index=True)

# Ordenar de mayor a menor Diferencia (rojo arriba, verde abajo)
top_impacto_pqsin = top_impacto_pqsin.sort_values(by='Diferencia', ascending=False)

top_impacto_pqsin['Etiqueta'] = top_impacto_pqsin['Categoría'] + " (OP: " + top_impacto_pqsin['OP Det'].astype(str) + ")"

top_impacto_pqsin['estado'] = top_impacto_pqsin['Diferencia'].apply(
    lambda x: 'Ahorro' if x < 0 else 'Sobrecosto'
)

fig_pqt_sinM = px.bar(
    top_impacto_pqsin,
    x='Diferencia',
    y='Etiqueta',
    orientation='h',
    color='estado',
    color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    template='plotly_dark'
)

# Esto elimina las líneas negras que dividen los segmentos dentro de la barra
fig_pqt_sinM.update_traces(marker_line_width=0) 

fig_pqt_sinM.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_pqt_sinM.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_pqt_sinM.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_pqt_sinM.show()

### Prelavado

In [30]:
df_prelavado = df_costos[df_costos['Proceso Producción'] == 'Prelavado'].copy()

# Definimos una columna de estado para tener el contexto de qué significa cada punto en la gráfica.
df_prelavado['estado'] = df_prelavado.apply(
    lambda 
    x: 'Ahorro' 
    if x['OS-Valor Unitario'] < x['costo_antes_iva'] 
    else 'Sobrecosto',
    axis=1
)

# Creamos la gráfica de bordado, lo que esta encima de la línea diagonal es ahorro y lo que esta debajo es sobrecosto
# El tamaño de cada punto representa el impacto económico

fig_pre= px.scatter(
    df_prelavado,
    x='OS-Valor Unitario',
    y='costo_antes_iva',
    color='Categoría',
    color_discrete_sequence=paleta_textil,
    hover_data=['OP Det', 'Nombre del Comercial', 'Costeo_Producto', 'OS-Valor Unitario', 'costo_antes_iva'],
    title='Prelavado',
)

min_val = min(df_prelavado['costo_antes_iva'].min(), df_prelavado['OS-Valor Unitario'].min())
max_val = max(df_prelavado['costo_antes_iva'].max(), df_prelavado['OS-Valor Unitario'].max())

fig_pre.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="pink", dash="dash")
)

fig_pre.update_traces(marker=dict(size=15)) 

fig_pre.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_pre.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1,
    title='Costo OS antes IVA'
)

fig_pre.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray',
    title='Costo antes IVA (costeado)'
)
fig_pre.show()

In [31]:
df_pre = df_costos[df_costos['Proceso Producción'] == 'Prelavado']

# Sumamos la diferencia por OP para que solo exista una barra por etiqueta
df_consolidado_pre = df_pre.groupby(['Categoría', 'OP Det']).agg({
    'Diferencia': 'sum'
}).reset_index()

# Ahora sí calculamos el impacto absoluto sobre el total de la OP
df_consolidado_pre['abs_impacto'] = df_consolidado_pre['Diferencia'].abs()

# Top 10 de las OPs con mayor impacto real
top_sobrecosto_pre = df_consolidado_pre.nlargest(10, 'Diferencia')
top_ahorro_pre = df_consolidado_pre.nsmallest(10, 'Diferencia')

# Unir ambos
top_impacto_pre = pd.concat([top_sobrecosto_pre, top_ahorro_pre], ignore_index=True)

# Ordenar de mayor a menor Diferencia (rojo arriba, verde abajo)
top_impacto_pre = top_impacto_pre.sort_values(by='Diferencia', ascending=False)

top_impacto_pre['Etiqueta'] = top_impacto_pre['Categoría'] + " (OP: " + top_impacto_pre['OP Det'].astype(str) + ")"

top_impacto_pre['estado'] = top_impacto_pre['Diferencia'].apply(
    lambda x: 'Ahorro' if x < 0 else 'Sobrecosto'
)

fig_prelavado = px.bar(
    top_impacto_pre,
    x='Diferencia',
    y='Etiqueta',
    orientation='h',
    color='estado',
    color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    template='plotly_dark'
)

# Esto elimina las líneas negras que dividen los segmentos dentro de la barra
fig_prelavado.update_traces(marker_line_width=0) 

fig_prelavado.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_prelavado.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_prelavado.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_prelavado.show()

### Corte y Confeccion

In [32]:
df_cyc = df_costos[df_costos['Proceso Producción'] == 'Corte y Confeccion'].copy()

# Definimos una columna de estado para tener el contexto de qué significa cada punto en la gráfica.
df_cyc['estado'] = df_cyc.apply(
    lambda 
    x: 'Ahorro' 
    if x['OS-Valor Unitario'] < x['costo_antes_iva'] 
    else 'Sobrecosto',
    axis=1
)

# Creamos la gráfica de bordado, lo que esta encima de la línea diagonal es ahorro y lo que esta debajo es sobrecosto
# El tamaño de cada punto representa el impacto económico

fig_cyc= px.scatter(
    df_cyc,
    x='OS-Valor Unitario',
    y='costo_antes_iva',
    color='Categoría',
    color_discrete_sequence=paleta_textil,
    hover_data=['OP Det', 'Nombre del Comercial', 'Costeo_Producto', 'OS-Valor Unitario', 'costo_antes_iva'],
    title='Corte y Confección',
)

min_val = min(df_cyc['costo_antes_iva'].min(), df_cyc['OS-Valor Unitario'].min())
max_val = max(df_cyc['costo_antes_iva'].max(), df_cyc['OS-Valor Unitario'].max())

fig_cyc.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="pink", dash="dash")
)

fig_cyc.update_traces(marker=dict(size=15)) 

fig_cyc.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_cyc.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1,
    title='Costo OS antes IVA'
)

fig_cyc.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray',
    title='Costo antes IVA (costeado)'
)
fig_cyc.show()

In [33]:
df_cycc = df_costos[df_costos['Proceso Producción'] == 'Corte y Confeccion']

# Sumamos la diferencia por OP para que solo exista una barra por etiqueta
df_consolidado_cyc = df_cycc.groupby(['Categoría', 'OP Det']).agg({
    'Diferencia': 'sum'
}).reset_index()

# Ahora sí calculamos el impacto absoluto sobre el total de la OP
df_consolidado_cyc['abs_impacto'] = df_consolidado_cyc['Diferencia'].abs()

top_sobrecosto_cyc = df_consolidado_cyc.nlargest(10, 'Diferencia')
top_ahorro_cyc = df_consolidado_cyc.nsmallest(10, 'Diferencia')

# Unir ambos
top_impacto_cyc = pd.concat([top_sobrecosto_cyc, top_ahorro_cyc], ignore_index=True)

# Ordenar de mayor a menor Diferencia (rojo arriba, verde abajo)
top_impacto_cyc = top_impacto_cyc.sort_values(by='Diferencia', ascending=False)

top_impacto_cyc['Etiqueta'] = top_impacto_cyc['Categoría'] + " (OP: " + top_impacto_cyc['OP Det'].astype(str) + ")"

top_impacto_cyc['estado'] = top_impacto_cyc['Diferencia'].apply(
    lambda x: 'Ahorro' if x < 0 else 'Sobrecosto'
)


fig_cycc = px.bar(
    top_impacto_cyc,
    x='Diferencia',
    y='Etiqueta',
    orientation='h',
    color='estado',
    color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    template='plotly_dark'
)

# Esto elimina las líneas negras que dividen los segmentos dentro de la barra
fig_cycc.update_traces(marker_line_width=0) 

fig_cycc.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_cycc.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_cycc.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)
fig_cycc.update_xaxes(tickformat='.2s')


### Paquete Completo

In [34]:
df_pqt_com = df_costos[df_costos['Proceso Producción'] == 'Paquete Completo'].copy()

# Definimos una columna de estado para tener el contexto de qué significa cada punto en la gráfica.
df_pqt_com['estado'] = df_pqt_com.apply(
    lambda 
    x: 'Ahorro' 
    if x['OS-Valor Unitario'] < x['costo_antes_iva'] 
    else 'Sobrecosto',
    axis=1
)

# Creamos la gráfica de bordado, lo que esta encima de la línea diagonal es ahorro y lo que esta debajo es sobrecosto
# El tamaño de cada punto representa el impacto económico

fig_pqt_com= px.scatter(
    df_pqt_com,
    x='OS-Valor Unitario',
    y='costo_antes_iva',
    color='Categoría',
    color_discrete_sequence=paleta_textil,
    hover_data=['OP Det', 'Nombre del Comercial', 'Costeo_Producto', 'OS-Valor Unitario', 'costo_antes_iva'],
    title='Paquete Completo',
)

min_val = min(df_pqt_com['costo_antes_iva'].min(), df_pqt_com['OS-Valor Unitario'].min())
max_val = max(df_pqt_com['costo_antes_iva'].max(), df_pqt_com['OS-Valor Unitario'].max())

fig_pqt_com.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="pink", dash="dash")
)

fig_pqt_com.update_traces(marker=dict(size=15)) 

fig_pqt_com.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_pqt_com.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1,
    title='Costo OS antes IVA'
)

fig_pqt_com.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray',
    title='Costo antes IVA (costeado)'
)
fig_pqt_com.show()

In [35]:

df_pqtcom = df_costos[df_costos['Proceso Producción'] == 'Paquete Completo']

# Sumamos la diferencia por OP para que solo exista una barra por etiqueta
df_consolidado_pqtcom = df_pqtcom.groupby(['Categoría', 'OP Det']).agg({
    'Diferencia': 'sum'
}).reset_index()

# Ahora sí calculamos el impacto absoluto sobre el total de la OP
df_consolidado_pqtcom['abs_impacto'] = df_consolidado_pqtcom['Diferencia'].abs()

top_sobrecosto_pqtcom = df_consolidado_pqtcom.nlargest(10, 'Diferencia')
top_ahorro_pqtcom = df_consolidado_pqtcom.nsmallest(10, 'Diferencia')

# Unir ambos
top_impacto_pqtcom = pd.concat([top_sobrecosto_pqtcom, top_ahorro_pqtcom], ignore_index=True)

# Ordenar de mayor a menor Diferencia (rojo arriba, verde abajo)
top_impacto_pqtcom = top_impacto_pqtcom.sort_values(by='Diferencia', ascending=False)

top_impacto_pqtcom['Etiqueta'] = top_impacto_pqtcom['Categoría'] + " (OP: " + top_impacto_pqtcom['OP Det'].astype(str) + ")"

top_impacto_pqtcom['estado'] = top_impacto_pqtcom['Diferencia'].apply(
    lambda x: 'Ahorro' if x < 0 else 'Sobrecosto'
)


fig_pqtcom = px.bar(
    top_impacto_pqtcom,
    x='Diferencia',
    y='Etiqueta',
    orientation='h',
    color='estado',
    color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    template='plotly_dark'
)

# Esto elimina las líneas negras que dividen los segmentos dentro de la barra
fig_pqtcom.update_traces(marker_line_width=0) 

fig_pqtcom.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_pqtcom.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_pqtcom.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

fig_pqtcom.show()

## Impacto por comercial

### Confección

In [36]:
# Filtramos por proceso de confección
df_confeccion = df_costos[df_costos['Proceso Producción'] == "Confección"]
df_confeccion['Tarea'] = df_confeccion['Categoría'].astype(str) + " (" + df_confeccion['Proceso Producción'].astype(str) + ")"


# Identificamos las tareas más costosas en el proceso de confección
top5_tareas = (df_confeccion.groupby('Tarea')['Diferencia']
               .apply(lambda x: x.abs().sum())  # suma de absolutos, no absoluto de la suma
               .sort_values(ascending=False)
               .head(8)
               .index.tolist())

df_plot = df_confeccion[df_confeccion['Tarea'].isin(top5_tareas)]

# Agrupamos por tarea (categoría + proceso) y por comercial, calculamos el promedio del costeo de cada uno
df_comparativa = df_plot.groupby(['Tarea', 'Nombre del Comercial'])['Diferencia'].mean().reset_index()
df_referencia = df_plot.groupby('Tarea')['Diferencia'].mean().reset_index()

# Hacemos la gráfica de barras del proceso de confección
fig_conf_comercial = px.bar(df_comparativa, 
             x='Diferencia', 
             y='Tarea', 
             color='Nombre del Comercial',
             barmode='group',
             orientation='h',
             text_auto='.0f',
             title='Confección',
             color_discrete_sequence=px.colors.qualitative.Bold)

fig_conf_comercial.update_layout(xaxis_title="Costo Promedio ($)", yaxis_title="", showlegend=True)

fig_conf_comercial.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_conf_comercial.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_conf_comercial.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)
fig_conf_comercial.show()

C:\Users\Laura\AppData\Local\Temp\ipykernel_23976\800916223.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_confeccion['Tarea'] = df_confeccion['Categoría'].astype(str) + " (" + df_confeccion['Proceso Producción'].astype(str) + ")"


### Bordado

In [37]:
# Filtramos por proceso de confección
df_bordado = df_costos[df_costos['Proceso Producción'] == "Bordado"]
df_bordado['Tarea'] = df_bordado['Categoría'].astype(str) + " (" + df_bordado['Proceso Producción'].astype(str) + ")"

# Identificamos las tareas más costosas en el proceso de confección
top5_tareas = (df_bordado.groupby('Tarea')['Diferencia valor unitario']
               .apply(lambda x: x.abs().sum())  # suma de absolutos, no absoluto de la suma
               .sort_values(ascending=False)
               .head(8)
               .index.tolist())

df_plot = df_bordado[df_bordado['Tarea'].isin(top5_tareas)]

# Agrupamos por tarea (categoría + proceso) y por comercial, calculamos el promedio del costeo de cada uno
df_comparativa_bordado = df_plot.groupby(['Tarea', 'Nombre del Comercial'])['Diferencia valor unitario'].mean().reset_index()
df_referencia_bordado = df_plot.groupby('Tarea')['Diferencia valor unitario'].mean().reset_index()

# Hacemos la gráfica de barras del proceso de confección
fig_bordado_comercial = px.bar(df_comparativa_bordado, 
             x='Diferencia valor unitario', 
             y='Tarea', 
             color='Nombre del Comercial',
             barmode='group',
             orientation='h',
             text_auto='.0f',
             title='Confección',
             color_discrete_sequence=px.colors.qualitative.Bold)

fig_bordado_comercial.update_layout(xaxis_title="Costo Promedio ($)", yaxis_title="", showlegend=True)

fig_bordado_comercial.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_bordado_comercial.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_bordado_comercial.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)
fig_bordado_comercial.show()

C:\Users\Laura\AppData\Local\Temp\ipykernel_23976\1513618330.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_bordado['Tarea'] = df_bordado['Categoría'].astype(str) + " (" + df_bordado['Proceso Producción'].astype(str) + ")"


### Sublimación

In [38]:
# Filtramos por proceso de confección
df_sublimacion = df_costos[df_costos['Proceso Producción'] == 'Sublimación']
df_sublimacion['Tarea'] = df_sublimacion['Categoría'].astype(str) + " (" + df_sublimacion['Proceso Producción'].astype(str) + ")"

# Identificamos las tareas más costosas en el proceso de confección
top5_tareas = (df_sublimacion.groupby('Tarea')['Diferencia valor unitario']
               .apply(lambda x: x.abs().sum())  # suma de absolutos, no absoluto de la suma
               .sort_values(ascending=False)
               .head(8)
               .index.tolist())

df_plot_sublimacion = df_sublimacion[df_sublimacion['Tarea'].isin(top5_tareas)]

# Agrupamos por tarea (categoría + proceso) y por comercial, calculamos el promedio del costeo de cada uno
df_comparativa_sublimacion = df_plot_sublimacion.groupby(['Tarea', 'Nombre del Comercial'])['Diferencia valor unitario'].mean().reset_index()
df_referencia_sublimacion = df_plot_sublimacion.groupby('Tarea')['Diferencia valor unitario'].mean().reset_index()

# Hacemos la gráfica de barras del proceso de confección
fig_sublimacion_comercial = px.bar(df_comparativa_sublimacion, 
             x='Diferencia valor unitario', 
             y='Tarea', 
             color='Nombre del Comercial',
             barmode='group',
             orientation='h',
             text_auto='.0f',
             title='Sublimación',
             color_discrete_sequence=px.colors.qualitative.Bold)

fig_sublimacion_comercial.update_layout(xaxis_title="Costo Promedio ($)", yaxis_title="", showlegend=True)

fig_sublimacion_comercial.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_sublimacion_comercial.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_sublimacion_comercial.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)
fig_sublimacion_comercial.show()

C:\Users\Laura\AppData\Local\Temp\ipykernel_23976\2877528957.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sublimacion['Tarea'] = df_sublimacion['Categoría'].astype(str) + " (" + df_sublimacion['Proceso Producción'].astype(str) + ")"


### Estampado

In [39]:
df_estampado = df_costos[df_costos['Proceso Producción'] == 'Estampado']
df_estampado['Tarea'] = df_estampado['Categoría'].astype(str) + " (" + df_estampado['Proceso Producción'].astype(str) + ")"


# Identificamos las tareas más costosas en el proceso de confección
top5_tareas = (df_estampado.groupby('Tarea')['Diferencia valor unitario']
               .apply(lambda x: x.abs().sum())  # suma de absolutos, no absoluto de la suma
               .sort_values(ascending=False)
               .head(8)
               .index.tolist())

df_plot_estampado = df_estampado[df_estampado['Tarea'].isin(top5_tareas)]

# Agrupamos por tarea (categoría + proceso) y por comercial, calculamos el promedio del costeo de cada uno
df_comparativa_estampado = df_plot_estampado.groupby(['Tarea', 'Nombre del Comercial'])['Diferencia valor unitario'].mean().reset_index()
df_referencia_estampado = df_plot_estampado.groupby('Tarea')['Diferencia valor unitario'].mean().reset_index()

# Hacemos la gráfica de barras del proceso de confección
fig_estampado_comercial = px.bar(df_comparativa_estampado, 
             x='Diferencia valor unitario', 
             y='Tarea', 
             color='Nombre del Comercial',
             barmode='group',
             orientation='h',
             text_auto='.0f',
             title='Estampado',
             color_discrete_sequence=px.colors.qualitative.Bold)

fig_estampado_comercial.update_layout(xaxis_title="Costo Promedio ($)", yaxis_title="", showlegend=True)

fig_estampado_comercial.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_estampado_comercial.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_estampado_comercial.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)
fig_estampado_comercial.show()

C:\Users\Laura\AppData\Local\Temp\ipykernel_23976\2022290529.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_estampado['Tarea'] = df_estampado['Categoría'].astype(str) + " (" + df_estampado['Proceso Producción'].astype(str) + ")"


### Paquete Completo Sin Marca

In [40]:
df_pqt_sin = df_costos[df_costos['Proceso Producción'] == 'Paquete Completo Sin Marca']
df_pqt_sin['Tarea'] = df_pqt_sin['Categoría'].astype(str) + " (" + df_pqt_sin['Proceso Producción'].astype(str) + ")"


# Identificamos las tareas más costosas en el proceso de confección
top5_tareas = (df_pqt_sin.groupby('Tarea')['Diferencia valor unitario']
               .apply(lambda x: x.abs().sum())  # suma de absolutos, no absoluto de la suma
               .sort_values(ascending=False)
               .head(8)
               .index.tolist())

df_plot_pqtsin = df_pqt_sin[df_pqt_sin['Tarea'].isin(top5_tareas)]

# Agrupamos por tarea (categoría + proceso) y por comercial, calculamos el promedio del costeo de cada uno
df_comparativa_pqtsin = df_plot_pqtsin.groupby(['Tarea', 'Nombre del Comercial'])['Diferencia valor unitario'].mean().reset_index()
df_referencia_pqtsin = df_plot_pqtsin.groupby('Tarea')['Diferencia valor unitario'].mean().reset_index()

# Hacemos la gráfica de barras del proceso de confección
fig_pqtsin_comercial = px.bar(df_comparativa_pqtsin, 
             x='Diferencia valor unitario', 
             y='Tarea', 
             color='Nombre del Comercial',
             barmode='group',
             orientation='h',
             text_auto='.0f',
             title='Paquete Completo Sin Marca',
             color_discrete_sequence=px.colors.qualitative.Bold)

fig_pqtsin_comercial.update_layout(xaxis_title="Costo Promedio ($)", yaxis_title="", showlegend=True)

fig_pqtsin_comercial.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_pqtsin_comercial.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_pqtsin_comercial.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)
fig_pqtsin_comercial.show()

C:\Users\Laura\AppData\Local\Temp\ipykernel_23976\2255326458.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pqt_sin['Tarea'] = df_pqt_sin['Categoría'].astype(str) + " (" + df_pqt_sin['Proceso Producción'].astype(str) + ")"


### Prelavado

In [41]:
df_prel = df_costos[df_costos['Proceso Producción'] == 'Prelavado']
df_prel['Tarea'] = df_prel['Categoría'].astype(str) + " (" + df_prel['Proceso Producción'].astype(str) + ")"

# Identificamos las tareas más costosas en el proceso de confección
top5_tareas = (df_prel.groupby('Tarea')['Diferencia valor unitario']
               .apply(lambda x: x.abs().sum())  # suma de absolutos, no absoluto de la suma
               .sort_values(ascending=False)
               .head(8)
               .index.tolist())

df_plot_prel = df_prel[df_prel['Tarea'].isin(top5_tareas)]

# Agrupamos por tarea (categoría + proceso) y por comercial, calculamos el promedio del costeo de cada uno
df_comparativa_prel = df_plot_prel.groupby(['Tarea', 'Nombre del Comercial'])['Diferencia valor unitario'].mean().reset_index()
df_referencia_prel = df_plot_prel.groupby('Tarea')['Diferencia valor unitario'].mean().reset_index()

# Hacemos la gráfica de barras del proceso de confección
fig_prelavado_comercial = px.bar(df_comparativa_prel, 
             x='Diferencia valor unitario', 
             y='Tarea', 
             color='Nombre del Comercial',
             barmode='group',
             orientation='h',
             text_auto='.0f',
             title='Paquete Completo Sin Marca',
             color_discrete_sequence=px.colors.qualitative.Bold)

fig_prelavado_comercial.update_layout(xaxis_title="Costo Promedio ($)", yaxis_title="", showlegend=True)

fig_prelavado_comercial.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_prelavado_comercial.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_prelavado_comercial.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)
fig_prelavado_comercial.show()

C:\Users\Laura\AppData\Local\Temp\ipykernel_23976\1468056171.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_prel['Tarea'] = df_prel['Categoría'].astype(str) + " (" + df_prel['Proceso Producción'].astype(str) + ")"


### Corte y Confeccion

In [42]:
df_cyc = df_costos[df_costos['Proceso Producción'] == 'Corte y Confeccion']
df_cyc['Tarea'] = df_cyc['Categoría'].astype(str) + " (" + df_cyc['Proceso Producción'].astype(str) + ")"


# Identificamos las tareas más costosas en el proceso de confección
top5_tareas = (df_cyc.groupby('Tarea')['Diferencia valor unitario']
               .apply(lambda x: x.abs().sum())  # suma de absolutos, no absoluto de la suma
               .sort_values(ascending=False)
               .head(8)
               .index.tolist())

df_plot_cyc = df_cyc[df_cyc['Tarea'].isin(top5_tareas)]

# Agrupamos por tarea (categoría + proceso) y por comercial, calculamos el promedio del costeo de cada uno
df_comparativa_cyc = df_plot_cyc.groupby(['Tarea', 'Nombre del Comercial'])['Diferencia valor unitario'].mean().reset_index()
df_referencia_cyc = df_plot_cyc.groupby('Tarea')['Diferencia valor unitario'].mean().reset_index()

# Hacemos la gráfica de barras del proceso de confección
fig_cyc_comercial = px.bar(df_comparativa_cyc, 
             x='Diferencia valor unitario', 
             y='Tarea', 
             color='Nombre del Comercial',
             barmode='group',
             orientation='h',
             text_auto='.0f',
             title='Paquete Completo Sin Marca',
             color_discrete_sequence=px.colors.qualitative.Bold)

fig_cyc_comercial.update_layout(xaxis_title="Costo Promedio ($)", yaxis_title="", showlegend=True)

fig_cyc_comercial.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_cyc_comercial.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_cyc_comercial.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)
fig_cyc_comercial.show()

C:\Users\Laura\AppData\Local\Temp\ipykernel_23976\3357795462.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cyc['Tarea'] = df_cyc['Categoría'].astype(str) + " (" + df_cyc['Proceso Producción'].astype(str) + ")"


### Paquete Completo

In [43]:
df_pqtcom = df_costos[df_costos['Proceso Producción'] == 'Paquete Completo']
df_pqtcom['Tarea'] = df_pqtcom['Categoría'].astype(str) + " (" + df_pqtcom['Proceso Producción'].astype(str) + ")"


# Identificamos las tareas más costosas en el proceso de confección
top5_tareas = (df_pqtcom.groupby('Tarea')['Diferencia valor unitario']
               .apply(lambda x: x.abs().sum())  # suma de absolutos, no absoluto de la suma
               .sort_values(ascending=False)
               .head(8)
               .index.tolist())

df_plot_pqtcom = df_pqtcom[df_pqtcom['Tarea'].isin(top5_tareas)]

# Agrupamos por tarea (categoría + proceso) y por comercial, calculamos el promedio del costeo de cada uno
df_comparativa_pqtcom = df_plot_pqtcom.groupby(['Tarea', 'Nombre del Comercial'])['Diferencia valor unitario'].mean().reset_index()
df_referencia_pqtcom = df_plot_pqtcom.groupby('Tarea')['Diferencia valor unitario'].mean().reset_index()

# Hacemos la gráfica de barras del proceso de confección
fig_pqtc_comercial = px.bar(df_comparativa_pqtcom, 
             x='Diferencia valor unitario', 
             y='Tarea', 
             color='Nombre del Comercial',
             barmode='group',
             orientation='h',
             text_auto='.0f',
             title='Paquete Completo Sin Marca',
             color_discrete_sequence=px.colors.qualitative.Bold)

fig_pqtc_comercial.update_layout(xaxis_title="Costo Promedio ($)", yaxis_title="", showlegend=True)

fig_pqtc_comercial.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
fig_pqtc_comercial.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

fig_pqtc_comercial.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)
fig_pqtc_comercial.show()

C:\Users\Laura\AppData\Local\Temp\ipykernel_23976\273474104.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pqtcom['Tarea'] = df_pqtcom['Categoría'].astype(str) + " (" + df_pqtcom['Proceso Producción'].astype(str) + ")"


# INFORME EN HTML

In [44]:
import base64

with open("logo.png", "rb") as image_file:
    logo_base64 = base64.b64encode(image_file.read()).decode()

In [45]:
import plotly
print(plotly.__version__)

6.6.0


In [46]:
import plotly.io as pio

for f_fig in [ figimpactoservicio,
              figboxplot, figboxtotal, fig_dif, fig_total, figcomercial,
              fig_confe, fig, fig_conf_comercial,
              fig_subl, fig_sub,  fig_sublimacion_comercial,
              fig_bord, fig_bordado, fig_bordado_comercial,
              fig_estampado, fig_est, fig_estampado_comercial,  
              fig_pqt_sin,  fig_pqt_sinM, fig_pqtsin_comercial,
              fig_pre, fig_prelavado, fig_prelavado_comercial,
              fig_cyc, fig_cycc, fig_cyc_comercial,
              fig_pqt_com, fig_pqtcom, fig_pqtc_comercial, fig_volumen, fig_ahorro, fig_valor, fig_cat_ahorro]:
    f_fig.update_layout(
        autosize=True,
        height=450,
        width=None,
        margin=dict(l=0, r=0, t=50, b=0)
    )

In [47]:
import os
from plotly.io import write_html

def export_reporte():
    output_dir = "./"
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, "index.html"), "w", encoding="utf-8") as f:

        # ── HEAD + ESTILOS ──────────────────────────────────────────
        f.write(f"""<!DOCTYPE html>
    <html lang='es'>
    <head>
    <meta charset='UTF-8'>
    <meta name='viewport' content='width=device-width, initial-scale=1.0'>
    <title>Informe de Costos</title>
    <style>
    
    /* Ocultar los radio buttons reales */
    input[type="radio"] {{
        display: none;
    }}

    /* Estilo de los "Botones" (Labels) */
                
    label {{
        display: inline-block;
        padding: 10px 25px;
        font-family: "Times New Roman", serif;
        font-size: 18px;
        background: #e0e0e0;
        cursor: pointer;
        border-radius: 10px 10px 0 0; /* Solo redondeado arriba */
        margin-right: 5px;
        transition: 0.3s;
    }}

    /* Efecto al pasar el mouse */
    label:hover {{
        background: #d0d0d0;
    }}

    /* Estilo del botón cuando está seleccionado */
    input[type="radio"]:checked + label {{
        background: #271B3F; /* Color oscuro para resaltar */
        color: white;
        font-weight: bold;
    }}

    /* Ocultar todos los contenidos por defecto */
    .tab-content {{
        display: none;
        padding: 20px;
        background: #ECE9FB;
        border-radius: 0 15px 15px 15px;
        box-shadow: 0 8px 16px rgba(0,0,0,0.1);
    }}

    /* Mostrar solo el contenido del radio button seleccionado */
    #tab-seleccion:checked ~ #content-seleccion,
    #tab-confeccion:checked ~ #content-confeccion,
    #tab-sublimacion:checked ~ #content-sublimacion,
    #tab-bordado:checked ~ #content-bordado,
    #tab-estampado:checked ~ #content-estampado,
    #tab-pqtsin:checked ~ #content-pqtsin,
    #tab-prelavado:checked ~ #content-prelavado,
    #tab-cyc:checked ~ #content-cyc,
    #tab-pqtc:checked ~ #content-pqtc
    {{
        display: block;
    }}
                
        body {{
            font-family: "Times New Roman", serif;
            margin: 40px;
            background-color: #ECE9FB;
            color: #673ded;
        }}
        .header {{
            position: relative;
            padding: 20px 0;
            border-bottom: 2px solid #ccc;
        }}
        .title {{
            font-size: 40px;
            font-weight: 800;
            color: #7166B0;
            text-align: center;
        }}
        .logo {{
            position: absolute;
            top: 50%;
            transform: translateY(-50%);
            height: 70px;
        }}
        .logo.left {{ left: 10px; }}
        .logo.right {{ right: 10px; }}
                
        .h2 {{
            font-size: 20px;
            color: #7166B0;
            margin-top: 50px;
        }}
        
        .row {{
            display: flex;
            gap: 30px;
            flex-wrap: wrap;
            margin-top: 30px;    
        }}
                
        .card .plotly-graph-div {{
            width: 100% !important;
            min-width: 0 !important;
            border-radius: 15px; 
            overflow: hidden;
        }}
                
        .card {{
            flex: 1;
            min-width: 400px;
            background-color: #F1EFFF;
            padding: 20px;
            width: 100%;
            border-radius: 10px;
            box-shadow: 0 10px 20px rgba(0,0,0,0.19), 0 6px 6px rgba(0,0,0,0.23);
        }}
        .card h3 {{
            text-align: center;
            font-size: 25px;
            color: #7166B0;
            font-weight: bold;
        }}
        .card p {{
            margin-top: 15px;
            font-size: 20px;
            color: #7166B0;
            font-weight: bold;
            text-align: center;
        }}
        
        .info-box p {{
            font-size: 20px;   
            color: #7166B0;
        }}
        
        .info-box h3 {{
            font-size: 25px;   
            color: #7166B0;
        }}
        
        .info-box ul {{
            padding-left: 20px;
            margin: 0;
        }}

        .info-box li {{
            font-size: 20px;
            color: #555;
            margin-bottom: 5px;
        }}
        
        /* Navegación Principal */
        .main-nav {{
            display: flex;
            justify-content: center;
            gap: 10px;
            margin-bottom: 30px;
            padding: 15px;
            background: #271B3F;
            border-radius: 15px;
        }}

        .main-nav label {{
            border-radius: 8px !important;
            background: #5c4eb3;
            color: white;
        }}

        /* Ocultar/Mostrar Páginas */
        .page-content {{
            display: none;
        }}

        #page-1:checked ~ #content-page-1,
        #page-2:checked ~ #content-page-2,
        #page-3:checked ~ #content-page-3{{
            display: block;
        }}
        
    </style>
    
    <script>
        // 1. Función maestra para redimensionar
        function ajustarGraficas() {{
            var graficas = document.querySelectorAll('.plotly-graph-div');
            graficas.forEach(function(g) {{
                Plotly.Plots.resize(g);
            }});
        }}

        // 2. Escuchar cambios en las pestañas (Radio Buttons)
        document.addEventListener('change', function(e) {{
            if (e.target.name === 'tabs' || e.target.name === 'main-pages') {{
            setTimeout(ajustarGraficas, 100);
            }}
        }});
        

        // 3. Ejecutar al cargar la página (por seguridad)
        window.addEventListener('load', function() {{
            setTimeout(ajustarGraficas, 300);
        }});

        // 4. Ejecutar si el usuario cambia el tamaño de la ventana del navegador
        window.addEventListener('resize', ajustarGraficas);
    </script>
                
</head>
<body>
        """)
        f.write(f"""
            <input type="radio" name="main-pages" id="page-1" checked>
            <input type="radio" name="main-pages" id="page-2">
            <input type="radio" name="main-pages" id="page-3">

            <div class="main-nav">
                <label for="page-1">📊 Informe de Costos</label>
                <label for="page-2">Procesos de producción</label>
                <label for="page-3">Análisis de costos</label>
            </div>

            <div class="page-content" id="content-page-1">
            
            <div class="header">

                <img src="data:image/png;base64,{logo_base64}" class="logo left">
                <div class="title">INFORME DE COSTOS MOLT</div>
                <img src="data:image/png;base64,{logo_base64}" class="logo right">

            </div>

            <div class="info-box">
            <h3>📌 CONTEXTO DEL ANÁLISIS</h3>
            <p>
            En el presente informe encontrará dos secciones principales. En la primera, denominada <b>"Procesos de producción"</b>, 
            se recopilan los datos de los últimos 6 meses relacionados con los costos de los servicios (valores antes de IVA), 
            incluyendo tanto el valor costeado por el área comercial como el valor realmente pagado en la OS. 
            Aquí encontrará gráficos diseñados para facilitar el análisis y apoyar la toma de decisiones.

            En la segunda sección, <b>"Informe de costos"</b>, se presentan inicialmente los gráficos correspondientes al análisis 
            del último año, incluyendo la identificación de los clientes con mayor facturación, así como los proveedores que 
            están cobrando por encima de lo que podría optimizarse con otras alternativas. 

            Adicionalmente, se incluyen visualizaciones del comportamiento histórico de variables clave, permitiendo entender 
            su evolución a lo largo del tiempo y facilitar la interpretación de la información. 

            Para una mejor lectura del informe, se recomienda tener en cuenta los siguientes conceptos:
            </p>

            <ul>
                <li><b>Diferencia de valor unitario:</b> Valor Unitario OS - Valor Unitario Costeado</li>
                <li><b>Diferencia de valor total:</b> Valor Total OS - Valor Total Costeado</li>
                <li>Si el valor de la OS es mayor al costeado → <b>subcosteo</b> para la compañía (y viceversa)</li>
            </ul>
            </div>
        """)
        
        f.write("</div>")
        
        f.write("""
        <div class="page-content" id="content-page-2">
            <div class="header">
            <div class="title">INFORME DE COSTOS DE SERVICIOS</div>
            </div>
        """)        
        
        f.write("<div class='row'>")
        # 🟢 Sobrecosteo
        f.write(f"""
            <div class='card'>
                <h2 style='color:#2E7D32; text-align:center;'>SOBRECOSTEO TOTAL</h2>
                    <div style='text-align:center; font-size:40px; color:#4CAF50; margin:10px 0;'>
        {ahorro_total_fmt}
                    </div>
                <p>Tomamos la diferencia de valor total y sumamos solo los valores negativos, así obtenemos el total general de sobrecosteo.</p>
            </div>
        """)

        # ⚫ Balance
        f.write(f"""
        <div class='card'>
                <h2 style='color:#424242; text-align:center;'>BALANCE NETO</h2>
                    <div style='text-align:center; font-size:40px; color:#000000; margin:10px 0;'>
        {balance_total_fmt}
                    </div>
                <p>Finalmente la suma de todos los valores tanto positivos como negativos nos da este valor correspondiente a un sobrecosteo, es decir, un ahorro para la compañía.</p>
        </div>
        """)

        # 🔴 Subcosteo
        f.write(f"""
            <div class='card'>
                <h2 style='color:#C62828; text-align:center;'>SUBCOSTEO TOTAL</h2>
                    <div style='text-align:center; font-size:40px; color:#E53935; margin:10px 0;'>
        {perdida_total_fmt}
                    </div>
                <p>Tomamos la diferencia de valor total y sumamos solo los valores positivos, así obtenemos el total general de subcosteo.</p>
            </div>
        """)

        f.write("</div>")
        
        f.write("<div class='row'>")

        f.write("<div class='card'><h3>IMPACTO ECONÓMICO TOTAL DE CADA PROCESO DE PRODUCCIÓN</h3>")
        write_html(figimpactoservicio, file=f, full_html=False,config={"responsive": True}, include_plotlyjs='cdn') 
        f.write("<p>Para este gráfico tomamos la diferencia de valor total, realizamos los cálculos correspondientes para obtener el valor total del impacto económico tanto sobrecosteado como subcosteado en cada proceso de producción, así mismo, tenemos el balance, si este se encuentra sobre 0 implica un Subcosteo y si se encuentra debajo de 0 un sobrecosteo</p></div>")

        f.write("</div>")
        
        
        # ── FILA 1 ──────────────────────────────────────────────────
        f.write("<div class='row'>")

        f.write("<div class='card'><h3>DIFERENCIA DE COSTO UNITARIO</h3>")
        write_html(figboxplot, file=f, full_html=False,config={"responsive": True}, include_plotlyjs='cdn') 
        f.write("<p>En este gráfico tomamos la diferencia de costo unitario para visualizar cómo varía esta diferencia en cada uno de los procesos de producción, los puntos por encima de 0 representan un subcosteo y los que están por debajo un sobrecosteo.</p></div>")

        f.write("</div>")
        
        # ── FILA 2 ──────────────────────────────────────────────────
        f.write("<div class='row'>")

        f.write("<div class='card'><h3>DIFERENCIA DE COSTO TOTAL</h3>")
        write_html(figboxtotal, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')
        f.write("<p>En este gráfico tomamos la diferencia de costo total para visualizar cómo varía esta diferencia en cada uno de los procesos de producción, los puntos por encima de 0 representan un subcosteo y los que están por debajo un sobrecosteo.</p></div>")

        f.write("</div>")

        #  ── FILA 3 ──────────────────────────────────────────────────
        
        f.write("<div class='row'>")

        f.write("<div class='card'><h3>TOP 5 CATEGORÍAS CON MAYOR IMPACTO ECONÓMICO CON RESPECTO A LA DIFERENCIA DE VALOR UNITARIO</h3>")
        write_html(fig_dif, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn') 
        f.write("<p>Este gráfico muestra la variación del costo unitario frente al promedio en cada categoría y proceso de producción. Nos permite visualizar las 5 categorías un una mayor variación en la diferencia de costo unitario</p></div>")

        f.write("</div>")
        
        #  ── FILA 4 ──────────────────────────────────────────────────
        
        f.write("<div class='row'>")

        f.write("<div class='card'><h3>TOP CATEGORÍAS CON PROCESO DE PRODUCCIÓN DE MÁS IMPACTO ECONÓMICO PARA LA COMPAÑÍA</h3>")
        write_html(fig_total, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn') 
        f.write("<p>Tomamos un top 10 de las categorías que más impacto económico han tenido en los últimos 6 meses en la compañía y en qué proceso de producción se presentó este impacto</p></div>")

        f.write("</div>")
        
        #
        
        f.write("<div class='row'>")

        f.write("<div class='card'><h3>MAPA DE CALOR PROMEDIO DE COSTEO POR CATEGORIA DE CADA COMERCIAL</h3>")
        write_html(figcomercial, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn') 
        f.write("<p>Sacamos el promedio de la diferencia de costo unitario por comercial, así sabemos en qué procesos de producción suelen sobrecostear o subcostear normalmente</p></div>")

        f.write("</div>")

        # ── Botones ──────────────────────────────────────────────────

        f.write("""
            <div class="tabs">

            <input type="radio" name="tabs" id="tab-seleccion" checked>
            <label for="tab-seleccion">SELECCIONAR 👉</label>
                
            <input type="radio" name="tabs" id="tab-confeccion">
            <label for="tab-confeccion">CONFECCIÓN</label>

            <input type="radio" name="tabs" id="tab-sublimacion">
            <label for="tab-sublimacion">SUBLIMACIÓN</label>

            <input type="radio" name="tabs" id="tab-bordado">
            <label for="tab-bordado">BORDADO</label>
            
            <input type="radio" name="tabs" id="tab-estampado">
            <label for="tab-estampado">ESTAMPADO</label>
            
            <input type="radio" name="tabs" id="tab-pqtsin">
            <label for="tab-pqtsin">PQT SIN MARCA</label>
            
            <input type="radio" name="tabs" id="tab-prelavado">
            <label for="tab-prelavado">PRELAVADO</label>
            
            <input type="radio" name="tabs" id="tab-cyc">
            <label for="tab-cyc">CORTE Y CONFECCION</label>

            <input type="radio" name="tabs" id="tab-pqtc">
            <label for="tab-pqtc">PAQUETE COMPLETO</label>

            <div class="tab-content" id="content-seleccion">
                <div style="text-align: center; padding: 50px; color: #7166B0;">
                    <h2 style="opacity: 0.5;">MOLT</h2>
                    <p>Selecciona un proceso arriba para ver los detalles.</p>
                </div>
            </div>
            
                               
            <div class="tab-content" id="content-confeccion">
                <div class='row'>
                    <div class='card'>
                        <h3>IMPACTO DE LA DIFERENCIA DE COSTO UNITARIO EN EL PROCESO DE CONFECCIÓN</h3>
        """)

        write_html(fig_confe, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
                        <p>En esta gráfica encontramos la visual de los ahorros y sobrecostos que tiene la diferencia de valores en el proceso de confección, los puntos por encima de la diagonal representan un sobrecosto y los que se encuentran por debajo un subcosteo</p>
                    </div>
                </div>
            </div>
        """)


        f.write("""
            <div class="tab-content" id="content-confeccion">
                <div class='row'>
                    <div class='card'>
                        <h3>IMPACTO ECONÓMICO DEL PROCESO DE CONFECCIÓN EN LA COMPAÑÍA</h3>
        """)

        write_html(fig, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
                        <p>En esta gráfica vemos el top 10 de las OP con mayor impacto económico para la compañía teniendo en cuenta la diferencia total entre el valor costeado y pagado.</p>
                    </div>
                </div>
            </div>
        """)
        
        # SUBLIMACIÓN
        
        f.write("""
            <div class="tab-content" id="content-sublimacion">
                <div class='row'>
                    <div class='card'>
                    <h3>IMPACTO DE LA DIFERENCIA DE COSTO UNITARIO EN EL PROCESO DE SUBLIMACIÓN</h3>
        """)

        write_html(fig_subl, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn') 

        f.write("""
                        <p>En esta gráfica encontramos la visual de los ahorros y sobrecostos que tiene la diferencia de valores en el proceso de sublimación, los puntos por encima de la diagonal representan un sobrecosto y los que se encuentran por debajo un subcosteo</p>
                    </div>
                </div>
            </div>
        """)


        f.write("""
            <div class="tab-content" id="content-sublimacion">
                <div class='row'>
                    <div class='card'>
                        <h3>IMPACTO ECONÓMICO DE LAS DIFERENCIAS DE COSTOS DEL PROCESO DE SUBLIMACIÓN EN LA COMPAÑÍA</h3>
        """)

        write_html(fig_sub, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
                        <p>En esta gráfica vemos las OP con mayor impacto económico para la compañía teniendo en cuenta la diferencia total entre el valor costeado y pagado.</p>
                    </div>
                </div>
            </div>
        """)

        #BORDADO

        f.write("""
            <div class="tab-content" id="content-bordado">
                <div class='row'>
                    <div class='card'>
                    <h3>IMPACTO DE LA DIFERENCIA DE COSTO UNITARIO EN EL PROCESO DE BORDADO</h3>
        """)

        write_html(fig_bordado, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn') 

        f.write("""
                        <p>En esta gráfica encontramos la visual de los ahorros y sobrecostos que tiene la diferencia de valores en el proceso de bordado, los puntos por encima de la diagonal representan un sobrecosto y los que se encuentran por debajo un subcosteo</p>
                    </div>
                </div>
            </div>
        """)

        f.write("""
            <div class="tab-content" id="content-bordado">
                <div class='row'>
                    <div class='card'>
                        <h3>IMPACTO ECONÓMICO DE LAS DIFERENCIAS DE COSTOS DEL PROCESO DE BORDADO EN LA COMPAÑÍA</h3>
        """)

        write_html(fig_bord, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
                        <p>En esta gráfica vemos las OP con mayor impacto económico para la compañía teniendo en cuenta la diferencia total entre el valor costeado y pagado.</p>
                    </div>
                </div>
            </div>
        """)
        
        # ESTAMPADO
        
        f.write("""
            <div class="tab-content" id="content-estampado">
                <div class='row'>
                    <div class='card'>
                    <h3>IMPACTO DE LA DIFERENCIA DE COSTO UNITARIO EN EL PROCESO DE ESTAMPADO</h3>
        """)

        write_html(fig_estampado, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn') 

        f.write("""
                        <p>En esta gráfica encontramos la visual de los ahorros y sobrecostos que tiene la diferencia de valores en el proceso de estampado, los puntos por encima de la diagonal representan un sobrecosto y los que se encuentran por debajo un subcosteo</p>
                    </div>
                </div>
            </div>
        """)
        f.write("""
            <div class="tab-content" id="content-estampado">
                <div class='row'>
                    <div class='card'>
                        <h3>IMPACTO ECONÓMICO DE LAS DIFERENCIAS DE COSTOS DEL PROCESO DE ESTAMPADO EN LA COMPAÑÍA</h3>
        """)

        write_html(fig_est, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
                        <p>En esta gráfica vemos las OP con mayor impacto económico para la compañía teniendo en cuenta la diferencia total entre el valor costeado y pagado.</p>
                    </div>
                </div>
            </div>
        """)        
               
        
        #PAQUETE COMPLETO SIN MARCA
        
        f.write("""
            <div class="tab-content" id="content-pqtsin">
                <div class='row'>
                    <div class='card'>
                    <h3>IMPACTO DE LA DIFERENCIA DE COSTO UNITARIO EN EL SERVICIO DE PAQUETE COMPLETO SIN MARCA</h3>
        """)

        write_html(fig_pqt_sin, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn') 

        f.write("""
                        <p>En esta gráfica encontramos la visual de los ahorros y sobrecostos que tiene la diferencia de valores en el servicio de paquete completo sin marca, los puntos por encima de la diagonal representan un sobrecosto y los que se encuentran por debajo un subcosteo</p>
                    </div>
                </div>
            </div>
        """)
        
        f.write("""
            <div class="tab-content" id="content-pqtsin">
                <div class='row'>
                    <div class='card'>
                        <h3>IMPACTO ECONÓMICO DE LAS DIFERENCIAS DE COSTOS DEL SERVICIO DE PAQUETE COMPLETO SIN MARCA EN LA COMPAÑÍA</h3>
        """)

        write_html(fig_pqt_sinM, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
                        <p>En esta gráfica vemos las OP con mayor impacto económico para la compañía teniendo en cuenta la diferencia total entre el valor costeado y pagado.</p>
                    </div>
                </div>
            </div>
        """) 
        
            
        
        # PRELAVADO
        
        f.write("""
            <div class="tab-content" id="content-prelavado">
                <div class='row'>
                    <div class='card'>
                    <h3>IMPACTO DE LA DIFERENCIA DE COSTO UNITARIO EN EL PROCESO DE PRELAVADO</h3>
        """)

        write_html(fig_pre, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn') 

        f.write("""
                        <p>En esta gráfica encontramos la visual de los ahorros y sobrecostos que tiene la diferencia de valores en el proceso de prelavao, los puntos por encima de la diagonal representan un sobrecosto y los que se encuentran por debajo un subcosteo</p>
                    </div>
                </div>
            </div>
        """)       
        
        f.write("""
            <div class="tab-content" id="content-prelavado">
                <div class='row'>
                    <div class='card'>
                        <h3>IMPACTO ECONÓMICO DE LAS DIFERENCIAS DE COSTOS DEL PROCESO DE PRELAVADO  EN LA COMPAÑÍA</h3>
        """)

        write_html(fig_prelavado, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
                        <p>En esta gráfica vemos las OP con mayor impacto económico para la compañía teniendo en cuenta la diferencia total entre el valor costeado y pagado.</p>
                    </div>
                </div>
            </div>
        """)        
          
 
        #CORTE Y CONFECCIÓN
 
        f.write("""
            <div class="tab-content" id="content-cyc">
                <div class='row'>
                    <div class='card'>
                    <h3>IMPACTO DE LA DIFERENCIA DE COSTO UNITARIO EN EL PROCESO DE CORTE Y CONFECCIÓN</h3>
        """)

        write_html(fig_cyc, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn') 

        f.write("""
                        <p>En esta gráfica encontramos la visual de los ahorros y sobrecostos que tiene la diferencia de valores en el proceso de corte y confección, los puntos por encima de la diagonal representan un sobrecosto y los que se encuentran por debajo un subcosteo</p>
                    </div>
                </div>
            </div>
        """)        
        
        f.write("""
            <div class="tab-content" id="content-cyc">
                <div class='row'>
                    <div class='card'>
                        <h3>IMPACTO ECONÓMICO DE LAS DIFERENCIAS DE COSTOS DEL PROCESO DE CORTE Y CONFECCIÓN EN LA COMPAÑÍA</h3>
        """)

        write_html(fig_cycc, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
                        <p>En esta gráfica vemos las OP con mayor impacto económico para la compañía teniendo en cuenta la diferencia total entre el valor costeado y pagado.</p>
                    </div>
                </div>
            </div>
        """)         
        
        # PAQUETE COMPLETO         
        
        
        f.write("""
            <div class="tab-content" id="content-pqtc">
                <div class='row'>
                    <div class='card'>
                    <h3>IMPACTO DE LA DIFERENCIA DE COSTO UNITARIO EN EL SERVICIO DE PAQUETE COMPLETO</h3>
        """)

        write_html(fig_pqt_com, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn') 

        f.write("""
                        <p>En esta gráfica encontramos la visual de los ahorros y sobrecostos que tiene la diferencia de valores en el servicio de paquete completo, los puntos por encima de la diagonal representan un sobrecosto y los que se encuentran por debajo un subcosteo</p>
                    </div>
                </div>
            </div>
        """)
        
        f.write("""
            <div class="tab-content" id="content-pqtc">
                <div class='row'>
                    <div class='card'>
                        <h3>IMPACTO ECONÓMICO DE LAS DIFERENCIAS DE COSTOS DEL SERVICIO DE PAQUETE COMPLETO EN LA COMPAÑÍA</h3>
        """)

        write_html(fig_pqtcom, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
                        <p>En esta gráfica vemos las OP con mayor impacto económico para la compañía teniendo en cuenta la diferencia total entre el valor costeado y pagado.</p>
                    </div>
                </div>
            </div>
        """)  
        f.write("</div>")
        f.write("</div>")
                
        f.write("""
            <div class="page-content" id="content-page-3">
                <div class="header">
                <div class="title">ANÁLISIS ESTRATÉGICO COSTOS</div>
            </div>
            
            <div class='row'>
                <div class='card'><h3>¿A QUÉ PROVEEDORES LES ESTAMOS PAGANDO SOBREPRECIO?</h3>
        """)

        write_html(fig_ahorro, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
            <p>Con este gráfico tipo pareto analizamos específicamente a qué proveedores normalmente les estamos pagando más de lo que podríamos pagar con otro proveedor</p>
            </div>
            </div>
            
            <div class='row'>
                <div class='card'><h3>¿EN QUÉ CLIENTES SE CONCENTRA LA MAYOR CARGA OPERATIVA?</h3>
        """)

        write_html(fig_volumen, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
            <p>Con este gráfico tipo pareto analizamos el impacto que tiene cada cliente en la cantidad de prendas que solicita, es decir, 
            los clientes a los cuales se está yendo el mayor esfuerzo de la mano de obra</p>
            </div>
            </div>

            <div class='row'>
                <div class='card'><h3>¿QUÉ CLIENTES REPRESENTAN EL MAYOR INGRESO PRESUPUESTADO A LA COMPAÑÍA?</h3>
        """)

        write_html(fig_valor, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
            <p>Con este gráfico tipo pareto analizamos los clientes en los que se concentra el mayor ingreso económico en la compañía</p>
            </div>
            </div>
            
            <div class='row'>
                <div class='card'><h3>¿EN QUÉ PRENDAS ESPECÍFICAS PODEMOS REDUCIR COSTOS?</h3>
        """)

        write_html(fig_cat_ahorro, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
            <p>Con este gráfico análizamos los tipos de prenda en los que normalmente gastamos más en el proceso de producción de lo que podríamos gastar realmente</p>
            </div>
            </div>

        """)
        
        f.write("</div>")
# Reporte con el nombre 'index' para poder dubirlo en github pages directamente.
    size_mb = os.path.getsize("index.html") / 1024 / 1024
    print(f"✅ Reporte exportado — Tamaño: {size_mb:.1f} MB")

export_reporte()


✅ Reporte exportado — Tamaño: 1.6 MB
